# Phase 5 — Final + LLM Verify (Qwen 2.5 7B Instruct)

Phase 5 = Phase 4 + a *grounded* LLM verification step:

```
retrieval → rerank → quality_partition → **LLM verify** → emit
```

The LLM sees the query + top-N reranked candidates (citation + 160-char text snippet)
and returns the subset it judges *directly* relevant. It can NEVER add a citation
that isn't in the candidate list (output is parsed against `candidate_set`).

## What's new vs Phase 4

| 영역 | Phase 4 | Phase 5 |
|---|---|---|
| Filter after rerank | quality partition (deprioritize SR-numeric) | **LLM verify** (Qwen 2.5 7B) — semantic precision filter |
| Diagnostic | per-query anchor hit/miss | + LLM keep rate + per-citation P/R split |
| Toggles | USE_TRANSLATE, USE_RERANKER, USE_COURT | + **USE_LLM** |

## Dataset to attach (in addition to phase 4 datasets)

`Qwen/Qwen2.5-7B-Instruct` (~15GB) — Kaggle Datasets 검색:
- "qwen2.5-7b-instruct" / "qwen 7b instruct" / 본인이 HF에서 받아 업로드
- 없으면 Internet ON 으로 첫 run 다운로드 (느림, ~5분)
- 정말 없으면 `USE_LLM = False` → phase 4 와 동일

## 기대 LB

Phase 4 0.25~0.45 → Phase 5 **0.30~0.50** (LLM verify 가 noise 빼주는 만큼)

## 시간 비용

- LLM verify: 쿼리당 ~5~10초 (Qwen 7B fp16, batch 1). 50 쿼리 = ~5~8분
- 노트북 12h 한도 안에 충분히 들어옴 (court 인덱스 캐시 후엔 총 ~15분)

In [ ]:
# Cell 1 — env, paths, pip
!pip install -q rank_bm25 sentence-transformers faiss-cpu

import re, csv, json, pickle, time, gc, sys, os
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

DATA = None
for cand in [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]:
    if cand.exists() and (cand / 'laws_de.csv').exists():
        DATA = cand; break
if DATA is None:
    raise RuntimeError(f'Data not found. /kaggle/input contents: {list(Path("/kaggle/input").iterdir())}')

WORK = Path('/kaggle/working')
CACHE = WORK / 'cache_phase3'
CACHE.mkdir(parents=True, exist_ok=True)
print(f'DATA: {DATA}')
print(f'WORK: {WORK}')

for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv', 'court_considerations.csv']:
    fp = DATA / p
    print(f'  {p:30s} {"ok" if fp.exists() else "MISSING"}  ({fp.stat().st_size/1e6:.1f} MB)' if fp.exists() else f'  {p:30s} MISSING')

# Toggles
USE_TRANSLATE = True       # NLLB EN→DE
USE_RERANKER  = True       # BGE-reranker-v2-m3
USE_COURT     = True       # court_considerations 2-stage (most expensive)
USE_LLM       = False      # Qwen verify — off by default; set True only if you've attached the model + have time


In [ ]:
import json

# Cell 2 — Multilingual abbreviation map (FR/IT → DE) — 1662 entries from Omnilex starter repo
MULTILANG_ABBR = {'OIAA': 'AAMV', 'OMAA': 'HVUV', 'DE-OMBat': 'AB-VASm', 'OBE-FINMA': 'ABV-FINMA', 'OAdo': 'AdoV', 'OAdoz': 'AdoV', 'SDA': 'ADS', 'ORAT': 'AEFV', 'OIAgr': 'AEV', 'LMCFA': 'AFZFG', 'LMCCE': 'AFZFG', 'OMCFA': 'AFZFV', 'OMCCE': 'AFZFV', 'LAVS': 'AHVG', 'RAVS': 'AHVV', 'OAVS': 'AHVV', 'LEAR': 'AIAG', 'LSAI': 'AIAG', 'OEAR': 'AIAV', 'OSAIn': 'AIAV', 'LEI': 'AIG', 'LStrI': 'AIG', 'OAccD': 'AkkBV', 'AccredO-LPsy': 'AkkredV-PsyG', 'OEAc-LPPsi': 'AkkredV-PsyG', 'LEDPP': 'ALBAG', 'LSRPP': 'ALBAG', 'OEDPP': 'ALBAV', 'OSRPP': 'ALBAV', 'OBat': 'AlgV', 'OSRA': 'V-NDA', 'OdA': 'AlkBestV', 'OTAl': 'AlkBestV', 'LAlc': 'AlkG', 'OAlc': 'AlkV', 'OAllerg': 'AllergV', 'OGEmol': 'AllgGebV', 'OgeEm': 'AllgGebV', 'OSites': 'AltlV', 'OSiti': 'AltlV', 'OSI-AC': 'ALV-IsV', 'OAMéd': 'AMBV', 'OAMed': 'AMBV', 'OEMéd': 'AMZV', 'OOMed': 'AMZV', 'OOrgA': 'AO', 'OEs': 'AO', 'OOS': 'AOV', 'OOV': 'AOV', 'LTr': 'ArG', 'LL': 'ArG', 'OLT 1': 'ArGV 1', 'OLL 1': 'ArGV 1', 'OLT 2': 'ArGV 2', 'OLL 2': 'ArGV 2', 'OLT 3': 'ArGV 3', 'OLL 3': 'ArGV 3', 'OLT 4': 'ArGV 4', 'OLL 4': 'ArGV 4', 'OLT 5': 'ArGV 5', 'OLL 5': 'ArGV 5', 'OITRV': 'ARPV', 'OTR 1': 'ARV 1', 'OLR 1': 'ARV 1', 'OTR 2': 'ARV 2', 'OLR 2': 'ARV 2', 'LSEtr': 'ASG', 'LSEst': 'ASG', 'Limpauto': 'AStG', 'LIAut': 'AStG', 'Oimpauto': 'AStV', 'OIAut': 'AStV', 'OSur-ASR': 'ASV-RAB', 'OS-ASR': 'ASV-RAB', 'LAsi': 'AsylG', 'OA 1': 'AsylV 1', 'OAsi 1': 'AsylV 1', 'OA 2': 'AsylV 2', 'OAsi 2': 'AsylV 2', 'OA 3': 'AsylV 3', 'OAsi 3': 'AsylV 3', 'LTrAlp': 'AtraG', 'LTAlp': 'AtraG', 'Otransa': 'AtraV', 'OTrAl': 'AtraV', 'LPGA': 'ATSG', 'OPGA': 'ATSV', 'RSTF': 'AufRBGer', 'RVTF': 'AufRBGer', 'OAsc': 'AufzV', 'OSAC': 'AuLaV', 'OAEs': 'AuLaV', 'LECCT': 'AVEG', 'LOCCL': 'AVEG', 'OFAC': 'AVFV', 'OFAD': 'AVFV', 'LSE': 'AVG', 'LC': 'AVG', 'LACI': 'AVIG', 'LADI': 'AVIG', 'OACI': 'AVIV', 'OADI': 'AVIV', 'OS': 'AVO', 'OS-FINMA': 'AVO-FINMA', 'OSE': 'AVV', 'OC': 'AVV', 'LDI': 'AwG', 'LDT': 'AZG', 'LDL': 'AZG', 'OLDT': 'AZGV', 'OLDL': 'AZGV', 'ODMA': 'BAlV', 'LB': 'BankG', 'LBCR': 'BankG', 'OB': 'BankV', 'OBCR': 'BankV', 'OTConst': 'BauAV', 'OLCostr': 'BauAV', 'LPCo': 'BauPG', 'LProdC': 'BauPG', 'OPCo': 'BauPV', 'OProdC': 'BauPV', 'LFPr': 'BBG', 'OFPr': 'BBV', 'LTI': 'BEG', 'LTCo': 'BEG', 'LHand': 'BehiG', 'LDis': 'BehiG', 'OHand': 'BehiV', 'ODis': 'BehiV', 'ONo-ASR': 'BekV-RAB', 'OC-ASR': 'BekV-RAB', 'LStup': 'BetmG', 'OCStup': 'BetmKV', 'OEPStup': 'BetmPV', 'OSPStup': 'BetmPV', 'OAStup': 'BetmSV', 'ODStup': 'BetmSV', 'OTStup-DFI': 'BetmVV-EDI', 'OEStup-DFI': 'BetmVV-EDI', 'OProP': 'BevSV', 'OPPop': 'BevSV', 'LFAIE': 'BewG', 'LAFE': 'BewG', 'OAIE': 'BewV', 'OAFE': 'BewV', 'LF-CLaH': 'BG-HAÜ', 'LF-CAA': 'BG-HAÜ', 'LTEJUS': 'BG-RVUS', 'LTAGSU': 'BG-RVUS', 'LAr': 'BGA', 'LDFR': 'BGBB', 'LMI': 'BGBM', 'LCITES': 'BGCITES', 'LF-CITES': 'BGCITES', 'RTF': 'BGerR', 'LFSP': 'BGF', 'LLCA': 'BGFA', 'LTF': 'BGG', 'LDEA': 'BGIAA', 'LSISA': 'BGIAA', 'LBCF': 'BGLE', 'LRFF': 'BGLE', 'LPPS': 'BGMD', 'LDPS': 'BGMD', 'LFPC': 'BGMK', 'LTrans': 'BGÖ', 'LTras': 'BGÖ', 'LEAC': 'BGRB', 'LIAC': 'BGRB', 'LJAr': 'BGS', 'LGD': 'BGS', 'LTN': 'BGSA', 'LLN': 'BGSA', 'LOST': 'BGST', 'LFSI': 'BGST', 'LFIF': 'BIFG', 'OBiG': 'BiGV', 'OIB-FINMA': 'BIV-FINMA', 'LCESF': 'BiZG', 'LCSFS': 'BiZG', 'LCMIF': 'BIZMB', 'OMPr': 'BMV', 'LMP': 'BöB', 'LAPub': 'BöB', 'OPDC': 'BPDV', 'OPDPers': 'BPDV', 'LSIP': 'BPI', 'LDP': 'BPR', 'LPSP': 'BPS', 'RNC': 'BSO', 'LSF': 'BStatG', 'LStat': 'BStatG', 'LIB': 'BStG', 'RAATPF': 'BStGerNR', 'ROATPF': 'BStGerNR', 'ROTPF': 'BStGerOR', 'RFPPF': 'BStKR', 'RSPPF': 'BStKR', 'OBioc': 'VBP', 'OBcarb': 'BTrV', 'LN': 'BüG', 'LCit': 'BüG', 'LSCPT': 'BÜPF', 'OREE': 'BURV', 'ORIS': 'BURV', 'OLN': 'VOSA', 'OCit': 'BüV', 'Cst.': 'BV', 'Cost.': 'BV', 'LPP': 'BVG', 'OPP 1': 'BVV 1', 'OPP 2': 'BVV 2', 'OPP 3': 'BVV 3', 'LIDV': 'BVVG', 'LDDV': 'BVVG', 'LMSI': 'BWIS', 'PCF': 'BZP', 'PC': 'BZP', 'OCart': 'CartV', 'LChim': 'ChemG', 'LPChim': 'ChemG', 'OEChim': 'ChemGebV', 'OEPChim': 'ChemGebV', 'OPICChim': 'ChemPICV', 'ORRChim': 'ChemRRV', 'ORRPChim': 'ChemRRV', 'OChim': 'ChemV', 'OPChim': 'ChemV', 'OCPCh': 'ChKV', 'OCCC': 'ChKV', 'CRTIF': 'COTIF 1980', 'LCaS-COVID-19': 'Covid-19-SBüG', 'LFiS-COVID-19': 'Covid-19-SBüG', 'OCyS': 'CSV', 'OCS': 'GGBV', 'CAC': 'CWÜ', 'OACP': 'CZV', 'OAut': 'CZV', 'ACCEES': 'ASEEGKVBSPMSA', 'ACSCSSS': 'ASEEGKVBSPMSA', 'LIFD': 'DBG', 'OSRP': 'DBV', 'OTDD': 'DBZV', 'LDes': 'DesG', 'ODes': 'DesV', 'OSEP': 'DGV', 'OSAP': 'DGV', 'RSA': 'DRA', 'RSE': 'DRA', 'LPD': 'DSG', 'OEng': 'DüV', 'OCon': 'DüV', 'OD-ASR': 'DV-RAB', 'OPD': 'DZV', 'LCdF': 'EBG', 'Lferr': 'EBG', 'OCF': 'EBV', 'Oferr': 'EBV', 'OITE-PT': 'EDAV-DS', 'OITE-PT-DFI': 'EDAV-DS-EDI', 'OITE-UE': 'EDAV-EU', 'OITE-UE-DFI': 'EDAV-EU-EDI', 'OITE-AC': 'EDAV-Ht', 'OITEAc': 'EDAV-Ht', 'OEES': 'EESV', 'O-HEFSM': 'EHSM-V', 'O-SUFSM': 'EHSM-V', 'OEmV': 'EichGebV', 'OEm-V': 'EichGebV', 'OIPR': 'EigVV', 'RIPP': 'EigVV', 'LIFM': 'EIMG', 'OIFM': 'EIMV', 'OO': 'EiV', 'OU': 'EiV', 'OCCP': 'EKBV', 'OCSC': 'EKBV', 'OESC-OFAG': 'EKV-BLW', 'OVC-UFAG': 'EKV-BLW', 'LIE': 'EleG', 'LPC': 'ELG', 'OPC-AVS/AI': 'ELV', 'LMETA': 'EMBAG', 'LMeCA': 'EMBAG', 'OMETA': 'EMBAV', 'OMeCA': 'EMBAV', 'LEmb': 'EmbG', 'OESMB': 'EmGvV', 'OACMB': 'EmGvV', 'LCMP': 'EMKG', 'OCMP': 'EMKV', 'OIMepe': 'EMmV', 'OSMisE': 'EMmV', 'CSDLLF': 'KSMG', 'CSDDDLF': 'KSMG', 'OEEE': 'EnEV', 'OEEne': 'EnEV', 'OEneR': 'EnFV', 'OPEn': 'EnFV', 'LIFSN': 'ENSIG', 'OIFSN': 'ENSIV', 'LDét': 'EntsG', 'LDist': 'EntsG', 'Odét': 'EntsV', 'ODist': 'EntsV', 'OAAE': 'EÖBV', 'OAPuE': 'EÖBV', 'OAAE-DFJP': 'EÖBV-EJPD', 'OAPuE-DFGP': 'EÖBV-EJPD', 'LAPG': 'EOG', 'LIPG': 'EOG', 'OAPG': 'EOV', 'OIPG': 'EOV', 'OFDEP': 'EPDFV', 'OFCIP': 'EPDFV', 'LDEP': 'EPDG', 'LCIP': 'EPDG', 'ODEP': 'EPDV', 'OCIP': 'EPDV', 'ODEP-DFI': 'EPDV-EDI', 'OCIP-DFI': 'EPDV-EDI', 'LEp': 'EpG', 'CBE 2000': 'EPÜ 2000', 'OEp': 'EpV', 'OFR': 'ERV', 'OFoP': 'ERV', 'CE-TAF': 'ERV-BVGer', 'OUC': 'ESV', 'OIConf': 'ESV', 'Oexpa': 'ExpaV', 'Oespa': 'ExpaV', 'LAFam': 'FamZG', 'OAFam': 'FamZV', 'OAFami': 'FamZV', 'OEV 4': 'FAV 4', 'OEA 4': 'FAV 4', 'OSVo': 'FDO', 'LSFin': 'FIDLEG', 'LSerFi': 'FIDLEG', 'OSFin': 'FIDLEV', 'OSerFi': 'FIDLEV', 'LERI': 'FIFG', 'LPRI': 'FIFG', 'OECin': 'FiFV', 'OPCin': 'FiFV', 'LIMF': 'FinfraG', 'LInFi': 'FinfraG', 'OIMF': 'FinfraV', 'OInFi': 'FinfraV', 'OIMF-FINMA': 'FinfraV-FINMA', 'OInFi-FINMA': 'FinfraV-FINMA', 'LEFin': 'FINIG', 'LIsFi': 'FINIG', 'OEFin': 'FINIV', 'OIsFi': 'FINIV', 'OEFin-FINMA': 'FINIV-FINMA', 'OIsFi-FINMA': 'FINIV-FINMA', 'Oém-FINMA': 'FINMA-GebV', 'Oem-FINMA': 'FINMA-GebV', 'OA-FINMA': 'FINMA-PV', 'LFINMA': 'FINMAG', 'OMPRI': 'FIPBV', 'LFiEl': 'FiREG', 'LAiSE': 'FiREG', 'CRSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'CSSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'LCF': 'FKG', 'OReCI': 'FKINV', 'OPRCI': 'FKINV', 'LFA': 'FLG', 'LAF': 'FLG', 'RFA': 'FLV', 'OA Fam': 'FLV', 'OLALA': 'FMBV', 'OLAlA': 'FMBV', 'LPMA': 'FMedG', 'LPAM': 'FMedG', 'OPMA': 'VLIp', 'OMP': 'VöB', 'LTC': 'FMG', 'OSALA': 'FMV', 'OsAlA': 'FMV', 'OFOrg': 'FOrgV', 'OAOrg': 'FOrgV', 'OH': 'FPV', 'OOra': 'FPV', 'OQICin': 'FQIV', 'OQIC': 'FQIV', 'ODE': 'FrSV', 'OEDA': 'FrSV', 'LFus': 'FusG', 'OMCo': 'FV', 'OMaeC': 'FV', 'OF-SCPT': 'FV-ÜPF', 'AECSDPC': 'ASEEGMF', 'ACSCS': 'ASEEGMF', 'OAG': 'GaGV', 'OAppG': 'GaGV', 'AGTDC': 'GATT', 'ORF': 'GBV', 'REmol-TAF': 'GebR-BVGer', 'TA-TAF': 'GebR-BVGer', 'REmol-TFB': 'GebR-PatGer', 'TA-TFB': 'GebR-PatGer', 'Olico': 'GeBüV', 'Olc': 'GeBüV', 'OE ArchF': 'GebV BAR', 'OE ARF': 'GebV BAR', 'OEmol-AFC': 'GebV ESTV', 'OEm-AFC': 'GebV ESTV', 'OELP': 'GebV SchKG', 'OTLEF': 'GebV SchKG', 'Oem-LEI': 'GebV-AIG', 'OEmol-LStrI': 'GebV-AIG', 'Oem-Acc': 'GebV-Akk', 'Oemo-Acc': 'GebV-Akk', 'OEmol-LTr': 'GebV-ArG', 'OEm-LL': 'GebV-ArG', 'OEmol-OFRO': 'GebV-ASTRA', 'OEmo-USTRA': 'GebV-ASTRA', 'OEmol-LSE': 'GebV-AVG', 'OEm-LC': 'GebV-AVG', 'OEmol-OFEV': 'GebV-BAFU', 'OE-UFAM': 'GebV-BAFU', 'OEmol-OFSPO': 'GebV-BASPO', 'OEm UFSPO': 'GebV-BASPO', 'OEmol-OFAC': 'GebV-BAZL', 'OEm-UFAC': 'GebV-BAZL', 'Oem-OFJ': 'GebV-BJ', 'Oem UFG': 'GebV-BJ', 'OEmol-OFAG': 'GebV-BLW', 'Ordinanza sulle tasse UFAG': 'GebV-BLW', 'OEmol-DFAE': 'GebV-EDA', 'OEm-DFAE': 'GebV-EDA', 'OEmol-DFI-BN': 'GebV-EDI-NBib', 'OEm-DFI-BN': 'GebV-EDI-NBib', 'OEmol-CMP': 'GebV-EMK', 'OEm-CMP': 'GebV-EMK', 'Oémol-En': 'GebV-En', 'OE-En': 'GebV-En', 'OEmol-ASF': 'GebV-ESA', 'OEm-AVF': 'GebV-ESA', 'OEmol-fedpol': 'GebV-fedpol', 'OEm-fedpol': 'GebV-fedpol', 'OREDT': 'GebV-FMG', 'OTST': 'GebV-FMG', 'OEmol-RC': 'GebV-HReg', 'OTa-IPI': 'GebV-IGE', 'OEmol-LCart': 'GebV-KG', 'OEm-LCart': 'GebV-KG', 'OEm-METAS': 'GebV-METAS', 'Oem-METAS': 'GebV-METAS', 'OEmol-BN': 'GebV-NBib', 'OEm-BN': 'GebV-NBib', 'OEmol-TP': 'GebV-öV', 'OEm-TP': 'GebV-öV', 'OEmol-Publ': 'GebV-Publ', 'OEm-Pub': 'GebV-Publ', 'OÉmol-CSA': 'GebV-SAR', 'OEm-CSA': 'GebV-SAR', 'OEmol-SEFRI': 'GebV-SBFI', 'OEm-SEFRI': 'GebV-SBFI', 'OE-RaP': 'GebV-StS', 'OEm-RaP': 'GebV-StS', 'OEmol-OHB': 'GebV-TPS', 'OEmSON': 'GebV-TPS', 'OEmol-DDPS': 'GebV-VBS', 'OEm-DDPS': 'GebV-VBS', 'OÉIPPSS': 'GebVO St', 'OSEIPSS': 'GebVO St', 'LGéo': 'GeoIG', 'LGI': 'GeoIG', 'OGéo': 'GeoIV', 'OGI': 'GeoIV', 'OGéo-swisstopo': 'GeoIV-swisstopo', 'OGI-swisstopo': 'GeoIV-swisstopo', 'OGéom': 'GeomV', 'Ogeom': 'GeomV', 'ONGéo': 'GeoNV', 'ONGeo': 'GeoNV', 'ORPSan': 'GesBAV', 'LPSan': 'GesBG', 'OCPSan': 'GesBKV', 'OSAS': 'GGBV', 'OCMD': 'GGUV', 'OMCont': 'GGUV', 'OIC-DFI': 'GgV-EDI', 'LCB': 'PAG', 'LBDI': 'GKG', 'ODVo': 'GKZ', 'OCPo': 'GKZ', 'LEg': 'GlG', 'LPar': 'GlG', 'OBPL': 'GLPV', 'RTFB': 'GR-PatGer', 'RI-COMCO': 'GR-WEKO', 'RCN': 'GRN', 'RCE': 'GRS', 'RCS': 'GRS', 'LEaux': 'GSchG', 'LPAc': 'GSchG', 'OEaux': 'GSchV', 'OPAc': 'GSchV', 'LGG': 'GTG', 'LIG': 'GTG', 'LAGH': 'GUMG', 'LEGU': 'GUMG', 'OAGH': 'GUMV', 'OEGU': 'GUMV', 'LTM': 'GüTG', 'OTM': 'GüTV', 'LTTM': 'GVVG', 'LTrasf': 'GVVG', 'LBA': 'GwG', 'LRD': 'GwG', 'OBA': 'GwV', 'ORD': 'GwV', 'OBA-OFDF': 'GwV-BAZG', 'ORD-UDSC': 'GwV-BAZG', 'OBA-DFJP': 'GwV-EJPD', 'ORD-DFGP': 'GwV-EJPD', 'OBA-CFMJ': 'GwV-ESBK', 'ORD-CFCG': 'GwV-ESBK', 'OBA-FINMA': 'GwV-FINMA', 'ORD-FINMA': 'GwV-FINMA', 'LTrD': 'HArG', 'LLD': 'HArG', 'OTrD': 'HArGV', 'OLLD': 'HArGV', 'OIPSD': 'HasLV', 'OIPSDA': 'HasLV', 'OIPSD-DEFR': 'HasLV-WBF', 'OIPSDA-DEFR': 'HasLV-WBF', 'OPFP-FINMA': 'HBEV-FINMA', 'LRH': 'HFG', 'LRUm': 'HFG', 'LEHE': 'HFKG', 'LPSU': 'HFKG', 'OMCR 20': 'HFMV 20', 'OPCR 20': 'HFMV 20', 'OMCR 22': 'HFMV 22', 'OPCR 22': 'HFMV 22', 'ORH': 'HFV', 'ORUm': 'HFV', 'LLGV': 'HGVAnG', 'LRAV': 'HGVAnG', 'OCBo': 'HHV', 'OCoL': 'HHV', 'CLaH96': 'HKsÜ', 'Convenzione dell’Aia sulla protezione dei minori': 'HKsÜ', 'OGOM': 'HKSV', 'OGOE': 'HKSV', 'LPTh': 'HMG', 'LATer': 'HMG', 'ORC': 'HRegV', 'OCCHE': 'HSBBV', 'OSCU': 'HSBBV', 'OMAV': 'HVA', 'OMAI': 'HVI', 'OMAINF': 'HVUV', 'OHyg': 'HyV', 'ORI': 'HyV', 'OIAM': 'IAMV', 'OSFPrHE': 'IBH-V', 'O-SIFPU': 'IBH-V', 'LSIS': 'SIaG', 'LSISpo': 'IBSG', 'OSIS': 'IBSV', 'OSISpo': 'IBSV', 'OMCC': 'VSFK', 'OCoCr': 'IBTV', 'OId-BDTA': 'IdTVD-V', 'OIBDTA': 'IdTVD-V', 'LIPPI': 'IFEG', 'LIPIn': 'IFEG', 'OIPI': 'IGE-OV', 'Oper-IPI': 'IGE-PersV', 'OPer-IPI': 'IGE-PersV', 'LIPI': 'IGEG', 'OAiR': 'InkHV', 'OAInc': 'InkHV', 'Oinv': 'InvV', 'OPICin': 'IPFiV', 'LDIP': 'IPRG', 'LISint': 'IQG', 'LIFI': 'IQG', 'RInfo-TFB': 'IR-PatGer', 'EIMP': 'IRSG', 'AIMP': 'IRSG', 'OEIMP': 'IRSV', 'OAIMP': 'IRSV', 'O-SI ABV': 'ISABV-V', 'O-SIAMV': 'ISABV-V', 'LSI': 'ISG', 'LSIn': 'ISG', 'O-SICAL': 'ISLK-V', 'O-SIFA': 'ISLK-V', 'OSIAgr': 'ISLV', 'OSIP-OFDF': 'IStrV-BAZG', 'OSIP-UDSC': 'IStrV-BAZG', 'OSAR': 'ISUV', 'OSIStr': 'ISUV', 'ODiv': 'IvDV', 'ODIV': 'IvDV', 'LAI': 'IVG', 'RAI': 'IVV', 'OAI': 'IVV', 'OSIAC': 'IVZV', 'O OFSPO J+S': 'J+S-V-BASPO', 'O-G+S-UFSPO': 'J+S-V-BASPO', 'LPMFJ': 'JSFVG', 'LPMFV': 'JSFVG', 'OPMFJ': 'JSFVV', 'OPMFV': 'JSFVV', 'LChP': 'JSG', 'LCP': 'JSG', 'DPMin': 'JStG', 'PPMin': 'JStPO', 'OChP': 'JSV', 'OCP': 'VKL', 'OFPC-FINMA': 'KAKV-FINMA', 'OFICol-FINMA': 'KAKV-FINMA', 'OAAcc': 'KBFHV', 'OACust': 'KBFHV', 'LENu': 'KEG', 'OENu': 'KEV', 'LEC': 'KFG', 'LPCu': 'KFG', 'OAVAuto': 'KFZV', 'OAuto': 'KFZV', 'LTBC': 'KGTG', 'OTBC': 'KGTV', 'OIBC': 'KGVV', 'OEBC': 'KGVV', 'LRCN': 'KHG', 'ORCN': 'KHV', 'LIC': 'KIG', 'LEEJ': 'KJFG', 'LPAG': 'KJFG', 'OEEJ': 'KJFV', 'OPAG': 'KJFV', 'LCC': 'KKG', 'OPCC': 'KKV', 'OICol': 'KKV', 'OPC-FINMA': 'KKV-FINMA', 'OICol-FINMA': 'KKV-FINMA', 'OClin': 'KlinV', 'OSRUm': 'KlinV', 'OClin-Dim': 'KlinV-Mep', 'OSRUm-Dmed': 'KlinV-Mep', 'OCI': 'KlV', 'OOCli': 'KlV', 'LFMG': 'KMG', 'LMB': 'KMG', 'OMG': 'KMV', 'OMB': 'KMV', 'OAOF': 'KOV', 'RUF': 'KOV', 'OCoo': 'KoVo', 'OCCRT': 'KoVo', 'OAMédcophy': 'KPAV', 'OMCF': 'KPAV', 'OCPF': 'KPFV', 'FP-TFB': 'KR-PatGer', 'SpG-TFB': 'KR-PatGer', 'OCré-FINMA': 'KreV-FINMA', 'OCre-FINMA': 'KreV-FINMA', 'OEMO': 'KRV', 'ORMT': 'KRV', 'Cst-GE': 'KV-GE', 'Cost.-GE': 'KV-GE', 'LSAMal': 'KVAG', 'LVAMal': 'KVAG', 'LAMal': 'KVG', 'OAMal': 'KVV', 'OPVA': 'LAfV', 'OPSAgr': 'LAfV', 'LRA': 'LBG', 'OAgrD': 'LDV', 'ODAgr': 'LDV', 'OLEl': 'LeV', 'LA': 'LFG', 'LNA': 'LFG', 'OSAv': 'LFV', 'ONA': 'LFV', 'OIBL': 'LGBV', 'OULiq': 'LGBV', 'OGN': 'LGeoIV', 'ODAlOUs': 'LGV', 'ODerr': 'LGV', 'OLiq': 'LiqV', 'OIDAl': 'LIV', 'OID': 'LIV', 'LDAl': 'LMG', 'LDerr': 'LMG', 'OIMLo': 'LMmV', 'OSML': 'LMmV', 'OELDAl': 'LMVV', 'OELDerr': 'LMVV', 'LBFA': 'LPG', 'LAAgr': 'LPG', 'OLRO-FINMA': 'LROV-FINMA', 'OPair': 'LRV', 'OIAt': 'LRV', 'LPPM': 'LSMG', 'OPPM': 'LSMV', 'OPB': 'LSV', 'OIF': 'LSV', 'OTrA': 'LTrV', 'CL': 'LugÜ', 'CLug': 'LugÜ', 'LAP': 'LVG', 'OMN': 'LVV', 'OMN-DDPS': 'LVV-VBS', 'LAgr': 'LwG', 'OAcCP': 'MAkkV', 'OAGio': 'MAkkV', 'OBMa': 'MaLV', 'ORMAp': 'MaLV', 'OMar-FINMA': 'MarV-FINMA', 'OMer-FINMA': 'MarV-FINMA', 'OMach': 'MaschV', 'OMacch': 'MaschV', 'OMat': 'MatV', 'OMAT-DDPS': 'MatV', 'ORM': 'MAV', 'OPart': 'MBV', 'OParC': 'MBV', 'OCMil': 'MCAV', 'OCDM': 'MCAV', 'ODqua': 'MeAV', 'OIQ': 'MeAV', 'ODqua-DFJP': 'MeAV-EJPD', 'OIQ-DFGP': 'MeAV-EJPD', 'LPMéd': 'MedBG', 'LPMed': 'MedBG', 'OPMéd': 'MedBV', 'OPMed': 'MedBV', 'ODim': 'MepV', 'ODmed': 'MepV', 'OSRM': 'MeQV', 'LMétr': 'MessG', 'LMetr': 'MessG', 'OIMes': 'MessMV', 'OStrM': 'MessMV', 'LMét': 'MetG', 'LMet': 'MetG', 'OMét': 'MetV', 'OMet': 'MetV', 'OSV': 'MFV', 'OSVM': 'MFV', 'LAAM': 'MG', 'LM': 'MG', 'OBCBA': 'MGwV', 'OURD': 'MGwV', 'LSIA': 'MIG', 'LSIM': 'MIG', 'OIMin': 'MindStV', 'OImM': 'MindStV', 'OMinTA': 'MinLV', 'Limpmin': 'MinöStG', 'LIOm': 'MinöStG', 'Oimpmin': 'MinöStV', 'OIOm': 'MinöStV', 'LUMin': 'MinVG', 'OUMin': 'MinVV', 'OCL': 'MiPV', 'OSIAr': 'MIV', 'OSIM': 'MIV', 'OCM ES': 'MiVo-HF', 'OERic-SSS': 'MiVo-HF', 'OJM': 'MJV', 'O-GM': 'MJV', 'OAMil': 'MLFV', 'OPCNP': 'MNKPV', 'LPM': 'MSchG', 'OPM': 'MSchV', 'CPM': 'MStG', 'PPM': 'MStP', 'OJPM': 'MStV', 'OGPM': 'MStV', 'O sur la monnaie': 'MünzV', 'OMon': 'MünzV', 'LAM': 'MVG', 'OAM': 'MVV', 'LTVA': 'MWSTG', 'LIVA': 'MWSTG', 'OTVA': 'MWSTV', 'OIVA': 'MWSTV', 'LFORTA': 'NAFG', 'LFOSTRA': 'NAFG', 'ONag': 'NagV', 'OANS': 'NARV', 'OPANS': 'NARV', 'LBN': 'NBG', 'LBNS': 'NBibG', 'OBNS': 'NBibV', 'LRens': 'NDG', 'LAIn': 'NDG', 'ORens': 'NDV', 'OAIn': 'NDV', 'OMBT': 'NEV', 'OPBT': 'NEV', 'OPU': 'NFSV', 'OPE': 'PAVO', 'LPN': 'NHG', 'OPN': 'NHV', 'LRNIS': 'NISSG', 'ORNI': 'NISV', 'OIBT': 'NIV', 'OPCN-OCDE': 'NKPV-OECD', 'OPCN-OCSE': 'NKPV-OECD', 'LVA': 'NSAG', 'LUSN': 'NSAG', 'OVA': 'NSAV', 'OUSN': 'NSAV', 'LRN': 'NSG', 'LSN': 'NSG', 'ORN': 'NSV', 'OSN': 'NSV', 'OARF': 'NZV', 'OARF-OFT': 'NZV-BAV', 'OARF-UFT': 'NZV-BAV', 'OHS-LP': 'OAV-SchKG', 'OAV-LEF': 'OAV-SchKG', 'LAO': 'OBG', 'LMD': 'OBG', 'OAO': 'VUB', 'OMD': 'OBV', 'OPub-FINMA': 'OffV-FINMA', 'LAVI': 'OHG', 'LAV': 'OHG', 'OAVI': 'OHV', 'CO': 'OR', 'OCRDP': 'ÖREBKV', 'OCRDPP': 'ÖREBKV', 'OdelO': 'OrFV', 'OTOr': 'OrFV', 'Org-OMP': 'Org-VöB', 'OOAPub': 'Org-VöB', 'Org ChF': 'OV-BK', 'OrgCaF': 'OV-BK', 'Org CF': 'OV-BR', 'OOrg-CF': 'OV-BR', 'Org DFAE': 'OV-EDA', 'OOrg-DFAE': 'OV-EDA', 'Org DFI': 'OV-EDI', 'OOrg-DFI': 'OV-EDI', 'Org DFF': 'OV-EFD', 'Org-DFF': 'OV-EFD', 'Org DFJP': 'OV-EJPD', 'Org-DFGP': 'OV-EJPD', 'Org LRH': 'OV-HFG', 'Org-LRUm': 'OV-HFG', 'Org DETEC': 'OV-UVEK', 'Org-DATEC': 'OV-UVEK', 'Org-DDPS': 'OV-VBS', 'OOrg-DDPS': 'OV-VBS', 'Org DEFR': 'OV-WBF', 'Org-DEFR': 'OV-WBF', 'LCBr': 'PAG', 'LParl': 'ParlG', 'OLPA': 'VABFP', 'Oparl': 'ParlVV', 'LPart': 'PartG', 'LUD': 'PartG', 'OPTP': 'PaRV', 'OPFP': 'PaRV', 'LBI': 'PatG', 'LTFB': 'PatGG', 'OParcs': 'PäV', 'OPar': 'PäV', 'OAMin': 'PAVO', 'OPTA': 'PAVV', 'LTV': 'PBG', 'OPD-EPF': 'PDV-ETH', 'OPDP-PF': 'PDV-ETH', 'LLG': 'PfG', 'LOF': 'PfG', 'OLG': 'PfV', 'OOF': 'PfV', 'OSaVé': 'PGesV', 'OSalV': 'PGesV', 'OSaVé-DEFR-DETEC': 'PGesV-WBF-UVEK', 'OSalV-DEFR-DATEC': 'PGesV-WBF-UVEK', 'ORPGAA': 'PGRELV', 'ORFGAA': 'PGRELV', 'OPha': 'PhaV', 'OFarm': 'PhaV', 'LOP': 'POG', 'LRFP': 'PrHG', 'LRDP': 'PrHG', 'LSPro': 'PrSG', 'OSPro': 'PrSV', 'ORRTP': 'PRTR-V', 'OPRTR': 'PRTR-V', 'OEPI': 'PSAV', 'ODPI': 'PSAV', 'OPPh': 'PSMV', 'OPF': 'VSB', 'OPsy': 'PsyBV', 'OPPsi': 'PsyBV', 'LPsy': 'PsyG', 'LPPsi': 'PsyG', 'LSPr': 'PüG', 'OPers-METAS': 'PV-METAS', 'OPersTF': 'PVBger', 'OPers-PDHH': 'PVFMH', 'OPers-PRA': 'PVFMH', 'OPers-PDHH-DDPS': 'PVFMH-VBS', 'OPers-PRA-DDPS': 'PVFMH-VBS', 'OPersT': 'PVGer', 'OPers-EPF': 'PVO-ETH', 'OPers PF': 'PVO-ETH', 'OPers-ServAS': 'PVO-TVS', 'OPers-SAT': 'PVO-TVS', 'OPers-PPOE': 'PVSPA', 'OPers-POE': 'PVSPA', 'OPers-PPOE-DDPS': 'PVSPA-VBS', 'OPers-POE-DDPS': 'PVSPA-VBS', 'OIS': 'QStV', 'OIFo': 'QStV', 'OQuaDu': 'QuNaV', 'OQuSo': 'QuNaV', 'R-COPA': 'R-UEK', 'LSR': 'RAG', 'OSRev': 'RAV', 'OEPC-FINMA': 'RelV-FINMA', 'OAPC-FINMA': 'RelV-FINMA', 'RCETF': 'ReRBGer', 'ORe-DFI': 'ResV-EDI', 'ORAMal-DFI': 'ResV-EDI', 'LHR': 'RHG', 'LArRa': 'RHG', 'OHR': 'RHV', 'OArRa': 'RHV', 'OSITC': 'RLSV', 'OITC': 'RLV', 'OrX': 'RöV', 'LAT': 'RPG', 'LPT': 'RPG', 'OAT': 'RPV', 'OPT': 'RPV', 'LRTV': 'RTVG', 'ORTV': 'RTVV', 'OR-AVS': 'RV-AHV', 'LOGA': 'RVOG', 'OLOGA': 'RVOV', 'LASEI': 'SAFIG', 'LASPI': 'SAFIG', 'OPTM': 'SAMV', 'OPLM': 'SAMV', 'OAGa': 'SaV', 'OASal': 'SaV', 'LCFF': 'SBBG', 'LFFS': 'SBBG', 'OMAS': 'SBMV', 'OMSC': 'SBMV', 'LP': 'SchKG', 'LEF': 'SchKG', 'LICa': 'SebG', 'LIFT': 'SebG', 'OICa': 'SebV', 'OIFT': 'SebV', 'OFDG': 'SEFV', 'OFDS': 'SEFV', 'OCâbles': 'SeilV', 'OFuni': 'SeilV', 'OASRE': 'SERV-V', 'OARE': 'SERV-V', 'LASRE': 'SERVG', 'LARE': 'SERVG', 'OPAAb': 'SGV', 'OPeM': 'SGV', 'LEIS': 'SIaG', 'LISDC': 'SIRG', 'CRUGBIN': 'BASEVKGNMD', 'ACSRUGBIN': 'BASEVKGNMD', 'ORIn': 'SnAV', 'ORim': 'SnAV', 'OMJ-DFJP': 'SPBV-EJPD', 'OCG-DFGP': 'SPBV-EJPD', 'OSLing': 'SpDV', 'LLC': 'SpG', 'LLing': 'SpG', 'LESp': 'SpoFöG', 'LPSpo': 'SpoFöG', 'OESp': 'SpoFöV', 'OPSpo': 'SpoFöV', 'LExpl': 'SprstG', 'LEspl': 'SprstG', 'OExpl': 'SprstV', 'OEspl': 'SprstV', 'OLang': 'SpV', 'OLing': 'SpV', 'LVP': 'SRVG', 'LPV': 'SRVG', 'LESE': 'SSchG', 'LSSE': 'SSchG', 'OESE': 'SSchV', 'OSSE': 'SSchV', 'OESE-DFI': 'SSchV-EDI', 'OSSE-DFI': 'SSchV-EDI', 'LNM': 'SSG', 'OSR': 'SSV', 'OSStr': 'SSV', 'LECF': 'StADG', 'LOA': 'StAG', 'LImA': 'StAG', 'LAAF': 'StAhiG', 'OAAF': 'StAhiV', 'OSOA': 'StAV', 'OImA': 'StAV', 'LOAP': 'StBOG', 'OASF': 'STEBV', 'LRCS': 'StFG', 'LCel': 'StFG', 'OPAM': 'StFV', 'OPIR': 'StFV', 'CP': 'StGB', 'LHID': 'StHG', 'LAID': 'StHG', 'OIMRI': 'StMmV', 'OSMRI': 'StMmV', 'CPP': 'StPO', 'LCJ': 'StReG', 'LCaGi': 'StReG', 'OCJ': 'StReV', 'OCaGi': 'StReV', 'LApEl': 'StromVG', 'LAEl': 'StromVG', 'OApEl': 'StromVV', 'OAEl': 'StromVV', 'LRaP': 'StSG', 'ORaP': 'StSV', 'LEnTR': 'STUG', 'LPTS': 'STUG', 'OEnTR': 'STUV', 'OTStr': 'STUV', 'LTRA': 'STVG', 'LTS': 'STVG', 'LSu': 'SuG', 'LRPL': 'SVAG', 'LTTP': 'SVAG', 'ORPL': 'SVAV', 'OTTP': 'SVAV', 'LCR': 'SVG', 'LCStr': 'SVG', 'OS LCart': 'SVKG', 'LPTab': 'TabPG', 'OPTab': 'TabPV', 'OETV 1': 'TAFV 1', 'OETV 2': 'TAFV 2', 'OETV 3': 'TAFV 3', 'OMédv': 'TAMV', 'OMvet': 'TAMV', 'OPBD': 'TBDV', 'OPPD': 'TBDV', 'LVPC': 'TEVG', 'LRVC': 'TEVG', 'OTRF': 'TGBV', 'OSSAn': 'TGDV', 'LETC': 'THG', 'LOTC': 'THG', 'OIMTh': 'TMmV', 'OSMT': 'TMmV', 'LTo': 'ToG', 'OTo': 'ToV', 'OFPT': 'TPFV', 'LTro': 'TrG', 'LIF': 'TrG', 'OFPAn': 'TSchAV', 'LPA': 'TSchG', 'LPAn': 'TSchG', 'OPAn': 'TSchV', 'LFE': 'TSG', 'LTab': 'TStG', 'LImT': 'TStG', 'OITab': 'TStV', 'OImT': 'TStV', 'LET': 'TUG', 'LATC': 'TUG', 'OServAS': 'TVSV', 'OSAT': 'TVSV', 'OPPPS': 'TwwV', 'OPPS': 'VMD', 'LACRE': 'UEG', 'LSgrI': 'UEG', 'OOPA': 'UEV', 'O-COPA': 'UEV', 'Règlement sur la pêche dans le lac Inférieur': 'Unterseefischereiordnung', 'Ordinamento della pesca nel Lago Inferiore': 'Unterseefischereiordnung', 'LTSM': 'UGüTG', 'LTMS': 'UGüTG', 'LIDE': 'UIDG', 'LIDI': 'UIDG', 'OIDE': 'UIDV', 'OIDI': 'UIDV', 'LPtra': 'ÜLG', 'LPTD': 'ÜLG', 'OPtra': 'ÜLV', 'OPTD': 'ÜLV', 'OUMR': 'UraM', 'MMRa': 'UraM', 'LDA': 'URG', 'ODAu': 'URV', 'LPE': 'USG', 'LPAmb': 'USG', 'LAA': 'UVG', 'LAINF': 'UVG', 'OEIE': 'UVPV', 'OEIA': 'UVPV', 'OLAA': 'UVV', 'OAINF': 'UVV', 'LCD': 'UWG', 'LCSl': 'UWG', 'OPersMil': 'V Mil Pers', 'OPers mil': 'V Mil Pers', 'OSEtr': 'V-ASG', 'OSEst': 'V-ASG', 'O-LERI': 'V-FIFG', 'O-LPRI': 'V-FIFG', 'O-LERI-DEFR': 'V-FIFG-WBF', 'O-LPRI-DEFR': 'V-FIFG-WBF', 'OLEH': 'V-GSG', 'OSOsp': 'V-GSG', 'O-LEHE': 'V-HFKG', 'O-LPSU': 'V-HFKG', 'O-STAC': 'V-LTDB', 'O-SIEs': 'V-NDA', 'O-LRNIS': 'V-NISSG', 'O-CNC-FPr': 'V-NQR-BB', 'O QNQ FP': 'V-NQR-BB', 'OCommcon-EPF': 'V-Schliko-ETH', 'O CommConc PF': 'V-Schliko-ETH', 'O-AQIS': 'V-SQWI', 'O-GQIS': 'V-SQWI', 'O-CP-CPM': 'V-StGB-MSt', 'OCP-CPM': 'V-StGB-MSt', 'OPNA': 'VABFP', 'OCDoc': 'VABK', 'RCDoc': 'VABK', 'OETHand': 'VAböV', 'ORTDis': 'VAböV', 'OISofCA': 'VABUA', 'OFSPE': 'VABUA', 'OCFH': 'VAEW', 'OIFI': 'VAEW', 'LSA': 'VAG', 'OMHSI': 'VAI', 'OMFSI': 'VAI', 'OIFC': 'VAK', 'OCFQE': 'VAK', 'OMéd': 'VAM', 'OM': 'VAM', 'OIMEC': 'VAMF', 'OMGC': 'VAMF', 'OMSVM': 'VAmFD', 'OIGE': 'VAMV', 'OSGS': 'VAMV', 'Odac': 'VAN', 'OEAC': 'VAN', 'OSRens': 'VAND', 'OVAIn': 'VAND', 'OLPS': 'VAPF', 'OQPN': 'VAPK', 'OEPIN': 'VAPK', 'OTAS': 'VASA', 'OTaRSi': 'VASA', 'OMBat': 'VASm', 'ONCR': 'VASR', 'OITEP': 'VATPE', 'OITIP': 'VATPE', 'OAHST': 'VATT', 'OATFS': 'VATT', 'OAAFM': 'VATV', 'OASAM': 'VATV', 'OAAFM-DDPS': 'VATV-VBS', 'OASAM-DDPS': 'VATV-VBS', 'ODPO': 'VAU', 'OAPO': 'VAU', 'OInstr pré': 'VAusb', 'OISP': 'VAusb', 'OInstr prém DDPS': 'VAusb-VBS', 'OISP-DDPS': 'VAusb-VBS', 'OMO': 'VAV', 'OMU': 'VAV', 'OMO-DDPS': 'VAV-VBS', 'OMU-DDPS': 'VAV-VBS', 'OLDI': 'VAwG', 'ODI': 'VID', 'OASMéd': 'VAZV', 'OOSM': 'VAZV', 'ODFR': 'VBB', 'Osol': 'VBBo', 'O suolo': 'VBBo', 'OLAr': 'VBGA', 'OLFP': 'VBGF', 'OTrans': 'VBGÖ', 'OTras': 'VBGÖ', 'OIFP': 'VBLN', 'OTUIC': 'VBNIB', 'ODO': 'VBO', 'OOC-SCPT': 'VBO-ÜPF', 'OThand': 'VböV', 'OTDis': 'VböV', 'OPBio': 'VBP', 'OIOP': 'VBPO', 'OOCOP': 'VBPO', 'O-OPers': 'VBPV', 'O-OPers-DFAE': 'VBPV-EDA', 'ORE I': 'VBR I', 'ONE I': 'VBR I', 'ORCSN': 'VBRK', 'ORTN': 'VBRK', 'OEMFP': 'VBSTB', 'OSMFP': 'VBSTB', 'OPSEnt': 'VBSV', 'OPSAz': 'VBSV', 'OAdma': 'VBVA', 'OAE-AF': 'VBVA', 'OGPCT': 'VBVV', 'OABCT': 'VBVV', 'OESN': 'VBWK', 'OCGIN': 'VBWK', 'OCITES': 'VCITES', 'O-CITES': 'VCITES', 'OME-SCPT': 'VD-ÜPF', 'OE-SCPT': 'VD-ÜPF', 'OODA': 'VDA', 'OODE': 'VDA', 'OTPSP': 'VDPS', 'ODPSP': 'VDPS', 'OEDRP-DFI': 'VDPV-EDI', 'OSDRP-DFI': 'VDPV-EDI', 'OCPD': 'VDSZ', 'OTNI': 'VDTI', 'OTDI': 'VDTI', 'OACA': 'VDZV', 'ODCA': 'VDZV', 'OLQE': 'VEAF', 'OLAE': 'VEAF', 'OIELFP': 'VEAGOG', 'OIEVFF': 'VEAGOG', 'OEFMP': 'VEBMP', 'OEE-VVT': 'VEE-PLS', 'OEE-AAT': 'VEE-PLS', 'ODF': 'VEJ', 'OBAF': 'VEJ', 'OGE': 'VEKF', 'OCGE': 'VEKF', 'OEmiA': 'VEL', 'OVotE': 'VEleS', 'OVE': 'VEleS', 'OMETr': 'VEMZ', 'OSTAC': 'VEMZ', 'OESS': 'VES', 'OIIS': 'VES', 'OCREPF': 'VETHBK', 'OCRPF': 'VETHBK', 'OCEl-PA': 'VeÜ-VwV', 'OCE-PA': 'VeÜ-VwV', 'OCEl-PCPP': 'VeÜ-ZSSV', 'OCE-PCPE': 'VeÜ-ZSSV', 'OEV': 'VEV', 'OMoD': 'VeVA', 'OTRif': 'VeVA', 'O E-VERA': 'VEVERA', 'Ordinanza E-VERA': 'VEVERA', 'OIMA': 'VLIb', 'OIMeA': 'VFAI', 'OAFA': 'VFAL', 'OOIT': 'VFAV', 'OPer-Fu': 'VFB-B', 'OLAFum': 'VFB-B', 'OPer-D': 'VFB-DB', 'OADAP': 'VFB-DB', 'OPer-H': 'VFB-G', 'OAS-O': 'VFB-G', 'OPer-B': 'VFB-H', 'OASL': 'VFB-H', 'OPer-Fl': 'VFB-K', 'OASPR': 'VFB-K', 'OPer-AH': 'VFB-LG', 'OASAOG': 'VFB-LG', 'OPer-P': 'VFB-S', 'OALPar': 'VFB-S', 'OPer-S': 'VFB-SB', 'OASSP': 'VFB-SB', 'OPer-Fo': 'VFB-W', 'OASEF': 'VFB-W', 'OVCC': 'VFBF', 'OMA': 'VFD', 'OILAE': 'VFlaW', 'OLDDA': 'VFlaW', 'OLCP': 'VFP', 'Oform': 'VFRR', 'Rform': 'VFRR', 'OSNA': 'VFSD', 'OSA': 'VFSD', 'OAF': 'VFV', 'LRCF': 'VG', 'LResp': 'VG', 'OFCoop': 'VGeK', 'RFCoop': 'VGeK', 'LTAF': 'VGG', 'FITAF': 'VGKE', 'TS-TAF': 'VGKE', 'RTAF': 'VGR', 'OJAr': 'VGS', 'OGD': 'VGS', 'OSPEX': 'VGSEB', 'OASAE': 'VGSEB', 'OEB': 'VGV', 'OIB': 'VGV', 'ODAlGM': 'VGVL', 'ODerrGM': 'VGVL', 'ORegBL': 'VGWR', 'OREA': 'VREG', 'OGOC': 'VHBT', 'OGOCC': 'VHBT', 'OCont': 'VHK', 'OHyPL': 'VHyMP', 'OIgPL': 'VHyMP', 'OHyPPr': 'VHyPrP', 'OIPPrim': 'VHyPrP', 'OHyAb': 'VHyS', 'OIgM': 'VHyS', 'ODIn': 'VID', 'OSIA': 'VIL', 'OILC': 'VILB', 'OSIC': 'VIM', 'OICoM': 'VIM', 'OCMI': 'VIMK', 'OIE': 'VIntA', 'OIntS': 'VIntA', 'OPPEtr': 'VIPaV', 'OIPPE': 'VIPaV', 'OSIS-SRC': 'VIS-NDB', 'OSIME-SIC': 'VIS-NDB', 'OISOS': 'VISOS', 'OVIS': 'VISV', 'OITPTh': 'VITH', 'OITAT': 'VITH', 'OIVS': 'VIVS', 'OCISF': 'ViZG', 'OCISC': 'ViZG', 'OCMIF': 'VIZMB', 'OJAR-FSTD': 'VJAR-FSTD', 'OACata': 'VKA', 'OLCC': 'VKKG', 'OCCEA': 'VKKL', 'OCoC': 'VKKL', 'OSPBC': 'VKKP', 'OSBC': 'VKKP', 'OCPre': 'VKL', 'OCSN': 'VKNS', 'OCos': 'VKos', 'OCTSE': 'VKOVA', 'OPFr': 'VKP', 'OCPPME': 'VKP-KMU', 'OCPPMI': 'VKP-KMU', 'OSSC': 'VKSD', 'Ocach': 'VKSWk', 'OCRCI': 'VKSWk', 'OFA-FINMA': 'VKV-FINMA', 'OCTrI': 'VKVöV', 'OCTrIn': 'VKVöV', 'OMDA': 'VKZ', 'OCA': 'VVK', 'OBNP': 'VLBE', 'ODPPE': 'VLBE', 'OBCF': 'VLE', 'ORFF': 'VLE', 'ORAgr': 'VLF', 'LCo': 'VlG', 'OECA': 'VLHb', 'OICA': 'VLHb', 'OOMA': 'VLIb', 'OPEA': 'VLIp', 'OCDA': 'VLKA', 'OCDAE': 'VLKA', 'ONAE': 'VLL', 'ODNA': 'VLL', 'ODAlOV': 'VLpH', 'ODOV': 'VLpH', 'ODAlAn': 'VLtH', 'ODOA': 'VLtH', 'OCo': 'VlV', 'OEMTP': 'VMAP', 'OSMLP': 'VMAP', 'OAMAS': 'VMBM', 'OAMM': 'VMBM', 'ODPS': 'VMD', 'OMi': 'VMDP', 'OOPSM': 'VMDP', 'OACAMIL': 'VMILAK', 'OACMIL': 'VMILAK', 'OAMC': 'VmKI', 'OMob': 'VMob', 'OSM': 'VMS', 'ONM': 'VMSch', 'OCM': 'VMSV', 'OCSM': 'VMSV', 'OBLF': 'VMWG', 'OLAL': 'VMWG', 'OOPT': 'VNEF', 'OORT': 'VNEF', 'OCNE': 'VNEK', 'OCAl': 'VNem', 'OIAl': 'VNem', 'OUS': 'VNF', 'OAPub': 'VöB', 'OCOV': 'VOCV', 'OSO': 'VOD', 'OOBE': 'VOEW', 'OOSE': 'VOEW', 'OOSG': 'VOGW', 'OCoR': 'VORA', 'OCoR-DFI': 'VORA-EDI', 'OTN': 'VOSA', 'OPoA': 'VPA', 'OPPE': 'VPA', 'ORCPP': 'VPABP', 'OPPCPers': 'VPABP', 'OSAss': 'VPAV', 'RPAss': 'VPAV', 'OTV': 'VPB', 'OPIE': 'VPeA', 'OPAR': 'VPEV', 'ORPAR': 'VPEV', 'OAPA': 'VPGA', 'ORInt': 'VPiB', 'OMP-OFEV': 'VpM-BAFU', 'OMF-UFAM': 'VpM-BAFU', 'OMP-OFAG': 'VpM-BLW', 'OMF-UFAG': 'VpM-BLW', 'OOP EPF': 'VPO ETH', 'OOP-PF': 'VPO ETH', 'OOPC': 'VPOB', 'OFipo': 'VPofi', 'OFiPo': 'VPofi', 'OLOP': 'VPOG', 'OOP': 'VPOG', 'ODP': 'VPR', 'OMAP': 'VPRG', 'ODFCLSI': 'VPRG', 'OPOVA': 'VPRH', 'OAOVA': 'VPRH', 'OPPr': 'VPrP', 'OPPrim': 'VPrP', 'OPSP': 'VPS', 'OCSP': 'VPSP', 'OEnB': 'VPV', 'RPB': 'VPV', 'OPAPIF': 'VPVE', 'ORPM': 'VPVK', 'ORPMUE': 'VPVKEU', 'RP-EPF 1': 'VR-ETH 1', 'RP-PF 1': 'VR-ETH 1', 'RP-EPF 2': 'VR-ETH 2', 'RP-PF 2': 'VR-ETH 2', 'OCBD': 'VRA', 'OCQO': 'VRA', 'RPEC': 'VRAB', 'RPIC': 'VRAB', 'ORSAE': 'VREG', 'OSCR': 'VRKD', 'ORésDAlan': 'VRLtH', 'ORDOA': 'VRLtH', 'OPR': 'VRP', 'ORCPL': 'VRSL', 'OReq': 'VRSL', 'ORA': 'VRV-L', 'ONCA': 'VRV-L', 'OPCF': 'VSB', 'OEMCN': 'VSBN', 'OSMCN': 'VSBN', 'OAbCV': 'VSFK', 'ODCS': 'VSFS', 'ODSRS': 'VSFS', 'OOCCR-OFROU': 'VSKV-ASTRA', 'OOCCS-USTR': 'VSKV-ASTRA', 'OMSA': 'VSL', 'OSMP': 'VSMS', 'OMSM': 'VSMS', 'ODiTr': 'VSoTr', 'ODiT': 'VSoTr', 'OPPBE': 'VSPA', 'OPBE': 'VSPA', 'OPESp': 'VSpoFöP', 'OPPSpo': 'VSpoFöP', 'OPPB': 'VSPS', 'ORS': 'VSR', 'ORSA': 'VSRL', 'OSJo': 'VSS', 'OSG': 'VSS', 'OOST': 'VST', 'OOSI': 'VST', 'ORCS': 'VStFG', 'ORCel': 'VStFG', 'LIA': 'VStG', 'LIP': 'VStG', 'DPA': 'VStrR', 'OIA': 'VStV', 'OIPrev': 'VStV', 'OSAA': 'VSUV', 'OSAI': 'VSUV', 'OFDPP': 'VSVB', 'OFDP': 'VSVB', 'OEIT': 'VSZV', 'OIET': 'VSZV', 'OCVM': 'VTE', 'OVF': 'VTE', 'OAP': 'VTM', 'OAAP': 'VTM', 'OSPA': 'VTNP', 'OSOAn': 'VTNP', 'OETV': 'VTS', 'OPAnAb': 'VTSchS', 'OPAnMac': 'VTSchS', 'OPAT': 'VWS', 'OPrTec': 'VtVtH', 'OOr': 'VUB', 'OOr-DEFR': 'VUB-WBF', 'OAO-DEFR': 'VUB-WBF', 'OFSPers': 'VUFB', 'OCPCP': 'VUKBK', 'OCIPP': 'VUKBK', 'OACM': 'VUM', 'OAAM': 'VUM', 'OSCPT': 'VÜPF', 'OPA': 'VUV', 'OPI': 'VUV', 'OVid-TP': 'VüV-ÖV', 'OVsor-TP': 'VüV-ÖV', 'OROPD': 'VUZPE', 'OROPS': 'VUZPE', 'OAA': 'VVA', 'OAE': 'VVA', 'OPC': 'VVAG', 'ODiC': 'VVAG', 'OOLDI': 'VVAwG', 'O-ODI-DFAE': 'VVAwG', 'OISec': 'VVE', 'OFIM': 'VVE', 'OLED': 'VVEA', 'OPSR': 'VVEA', 'LCA': 'VVG', 'OTeA': 'VVK', 'OCA-DFI': 'VVK-EDI', 'OTeA-DFI': 'VVK-EDI', 'OMAH': 'VVMH', 'OMPAH': 'VVMH', 'OOUS': 'VVNF', 'OUUS': 'VVNF', 'OST-SCPT': 'VVS-ÜPF', 'OPSE': 'VVSG', 'OPreS': 'VVSG', 'OERE': 'VVWAL', 'OEAE': 'VVWAL', 'LALM': 'VWBG', 'LMAM': 'VWBG', 'OLCAP': 'VWEG', 'OFSI': 'VWEV', 'OCMSD': 'VWEV', 'OSS': 'VWL', 'OAEP': 'VWLV', 'OPATE': 'VWS', 'PA': 'VwVG', 'OASA': 'VZAE', 'LGEL': 'VZEG', 'LPIL': 'VZEG', 'OSCSE': 'VZertES', 'OFiEle': 'VZertES', 'ORFI': 'VZG', 'RFF': 'VZG', 'ONCAF': 'VZSchB', 'OAC': 'VZV', 'OASM': 'VZVM', 'OAVM': 'VZVM', 'LFo': 'WaG', 'OFo': 'WaV', 'LFCo': 'WeBiG', 'OFCo': 'WeBiV', 'OEPL': 'WEFV', 'OPPA': 'WEFV', 'LCAP': 'WEG', 'LArm': 'WG', 'LAMO': 'WHG', 'LTEO': 'WPEG', 'OTEO': 'WPEV', 'OIRH': 'WResV', 'OREI': 'WResV', 'LFH': 'WRG', 'LUFI': 'WRG', 'OFH': 'WRV', 'OUFI': 'WRV', 'LPAP': 'WSchG', 'LPSt': 'WSchG', 'OPAP': 'WSchV', 'OPSt': 'WSchV', 'OMT': 'WTO', 'OArm': 'WV', 'LUMMP': 'WZG', 'LUMP': 'WZG', 'RDE': 'WZV', 'ODA': 'WZV', 'OROEM': 'WZVV', 'ORUAM': 'WZVV', 'LUsC': 'ZAG', 'LCoe': 'ZAG', 'LFisE': 'ZBstG', 'LFR': 'ZBstG', 'LSC': 'ZDG', 'ODSC': 'ZDUeV', 'OSCi': 'ZDV', 'OSCi-DEFR': 'ZDV-WBF', 'LDIF': 'ZEBG', 'LSIF': 'ZEBG', 'LOC': 'ZentG', 'LUC': 'ZentG', 'SCSE': 'ZertES', 'FiEle': 'ZertES', 'Ltém': 'ZeugSG', 'LPTes': 'ZeugSG', 'OTém': 'ZeugSV', 'OPTes': 'ZeugSV', 'OADou': 'ZEV', 'OADo': 'ZEV', 'LD': 'ZG', 'CC': 'ZGB', 'LCPI': 'ZISG', 'OCMétr': 'ZMessV', 'OCMetr': 'ZMessV', 'CPC': 'ZPO', 'CCoop-ESF': 'ZSAV-BiZ', 'CColl-SFS': 'ZSAV-BiZ', 'CCoop-HE': 'ZSAV-HS', 'ConSU': 'ZSAV-HS', 'OAASF': 'ZSTEBV', 'OEEC': 'ZStGV', 'OESC': 'ZStGV', 'OEC': 'ZStV', 'OSC': 'ZStV', 'OPCi': 'ZSV', 'LTaD': 'ZTG', 'LTD': 'ZTG', 'LAS': 'ZUG', 'OComp-OSPro': 'ZustV-PrSV', 'OAdd': 'ZuV', 'OD': 'ZV', 'OD-OFDF': 'ZV-BAZG', 'OD-UDSC': 'ZV-BAZG', 'OD-DFF': 'ZV-EFD', 'OA-DFJP': 'ZV-EJPD', 'OA-DFGP': 'ZV-EJPD', 'LRS': 'ZWG', 'LASec': 'ZWG', 'ORSec': 'ZWV', 'OASec': 'ZWV'}
print(f'multilang entries: {len(MULTILANG_ABBR)}')

In [ ]:
# Cell 3 — citation parsing, normalization, corpus index, granularity filter
ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)$'
)

def parse_article(c):
    m = ART_PATTERN.match(c.strip()); return m.groupdict() if m else None

def normalize_whitespace(c):
    s = re.sub(r'\s+', ' ', c.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    s = re.sub(r'\bZiff\.(?=\d)', 'Ziff. ', s)
    return s

def expand_multilang(c):
    out = [c]
    m = parse_article(c)
    if m and m['code'] in MULTILANG_ABBR:
        code_de = MULTILANG_ABBR[m['code']]
        out.append(c[:c.rfind(m['code'])] + code_de)
    return out

print('Loading laws & court citation sets ...')
laws_df = pd.read_csv(DATA / 'laws_de.csv')
laws_cits = set(laws_df['citation'].astype(str))
court_cits = set()
with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
CORPUS = laws_cits | court_cits
print(f'corpus citations: laws={len(laws_cits):,}  court={len(court_cits):,}  combined={len(CORPUS):,}')

PARENT_TO_CHILDREN = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    parent = f"Art. {m.group('art')} {m.group('code')}"
    if c != parent:
        PARENT_TO_CHILDREN[parent].append(c)
print(f'parents with children: {len(PARENT_TO_CHILDREN):,}')

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in CORPUS: return [c]
    for v in expand_multilang(c):
        if v != c and v in CORPUS: return [v]
        if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    if c in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[c])
    m = parse_article(c)
    if m:
        parent = f"Art. {m['art']} {m['code']}"
        if parent != c and parent in CORPUS: return [parent]
        for v in expand_multilang(parent):
            if v != c and v in CORPUS: return [v]
            if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    return []

def granularity_filter(cits):
    cs = set(cits); keep=[]
    for c in cits:
        kids = PARENT_TO_CHILDREN.get(c)
        if kids and any(k in cs for k in kids): continue
        keep.append(c)
    return keep

def dedup(cits):
    seen=set(); out=[]
    for c in cits:
        if c not in seen: seen.add(c); out.append(c)
    return out

# sanity: train resolve rate
train_df = pd.read_csv(DATA / 'train.csv')
def split_cits(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [t.strip() for t in s.split(';') if t.strip()]
total=resolved=0
for _, r in train_df.iterrows():
    for g in split_cits(r['gold_citations']):
        total+=1
        if resolve_against_corpus(g): resolved+=1
print(f'TRAIN gold resolve: {resolved}/{total} = {resolved/max(1,total):.4f}')
assert resolved/max(1,total) > 0.95, 'normalization regressed below 95% — investigate'


In [ ]:
# Cell 4 — shared BGE-M3 encoder (used by laws + court + query encoding)
import torch
from sentence_transformers import SentenceTransformer

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

BGE_M3_PATHS = [
    '/kaggle/input/bge-m3/bge-m3',
    '/kaggle/input/bge-m3',
    '/kaggle/input/baai-bge-m3',
    'BAAI/bge-m3',
]

ENCODER = None
for p in BGE_M3_PATHS:
    try:
        print(f'trying {p} ...')
        ENCODER = SentenceTransformer(p)
        print(f'BGE-M3 loaded from {p}')
        break
    except Exception as e:
        print(f'  failed: {type(e).__name__}: {str(e)[:120]}')

if ENCODER is None:
    raise RuntimeError('BGE-M3 not available. Attach BAAI/bge-m3 dataset or enable Internet.')

if torch.cuda.is_available():
    ENCODER = ENCODER.to('cuda')
    try: ENCODER.half()
    except Exception: pass

def encode(texts, batch_size=64, show_progress=False):
    return ENCODER.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                          show_progress_bar=show_progress, convert_to_numpy=True).astype('float32')

def encode_query(q):
    return encode([q], batch_size=1)


In [ ]:
# Cell 5 — laws_de.csv BM25 + BGE-M3 dense (cached)
import faiss
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9]+')
def tokenize(text):
    if not isinstance(text, str): return []
    return [t.lower() for t in TOKEN_RE.findall(text)]

laws_citations = laws_df['citation'].astype(str).tolist()
laws_texts = laws_df['text'].fillna('').astype(str).tolist()

bm25_path = CACHE / 'laws_bm25.pkl'
if bm25_path.exists():
    print('Loading cached laws BM25 ...')
    with open(bm25_path, 'rb') as f: bm25 = pickle.load(f)
else:
    print('Tokenizing & building BM25 (~1 min) ...')
    t0=time.time(); tokenized=[tokenize(t) for t in laws_texts]
    bm25 = BM25Okapi(tokenized)
    with open(bm25_path, 'wb') as f: pickle.dump(bm25, f)
    print(f'  BM25 built in {time.time()-t0:.1f}s')

faiss_path = CACHE / 'laws_dense.faiss'
if faiss_path.exists():
    print('Loading cached laws FAISS ...')
    laws_dense = faiss.read_index(str(faiss_path))
else:
    print(f'Encoding {len(laws_texts):,} laws rows with BGE-M3 (~5~10 min on T4) ...')
    t0=time.time()
    emb = encode(laws_texts, batch_size=64, show_progress=True)
    print(f'  encoded in {time.time()-t0:.1f}s  shape={emb.shape}')
    laws_dense = faiss.IndexFlatIP(emb.shape[1])
    laws_dense.add(emb)
    faiss.write_index(laws_dense, str(faiss_path))
    del emb; gc.collect()
print(f'laws FAISS ready: ntotal={laws_dense.ntotal}  d={laws_dense.d}')


In [ ]:
# Cell 6 — court_considerations 2-stage indexing (judgment → E.-level)
# This is the heaviest cell. Disable with USE_COURT=False above if time-constrained.
# Note: e_texts (truncated to 400 chars) are persisted so the reranker has something
# to score court citations against. Without this, phase 3 over phase 2 yields zero
# improvement on the 41% of gold that is BGE/BGer.
RE_BGE  = re.compile(r'^BGE\s+\d+\s+[IVX]+\s+\d+')
RE_NEW  = re.compile(r'^\d+[A-Za-z]_\d+/\d+')
RE_LEG  = re.compile(r'^([A-Z])\s*\d+/\d+\s+\d{2}\.\d{2}\.\d{4}')
RE_LEGC = re.compile(r'^\d+[A-Z]\.\d+/\d+\s+\d{2}\.\d{2}\.\d{4}')

def judgment_id(cit):
    for r in (RE_BGE, RE_NEW, RE_LEG, RE_LEGC):
        m = r.match(cit)
        if m: return m.group(0)
    return None

court_meta_path = CACHE / 'court_meta.pkl'
court_jfaiss    = CACHE / 'court_j.faiss'
court_efaiss    = CACHE / 'court_e.faiss'

E_TEXT_TRUNC = 400  # chars — bge-reranker has 512-token window

if USE_COURT and court_meta_path.exists() and court_jfaiss.exists() and court_efaiss.exists():
    print('Loading cached court indexes ...')
    with open(court_meta_path, 'rb') as f: court_meta = pickle.load(f)
    j_dense = faiss.read_index(str(court_jfaiss))
    e_dense = faiss.read_index(str(court_efaiss))
    # backward compatible: if cache was built without e_texts_trunc, stream-rebuild that field only
    if 'e_texts_trunc' not in court_meta:
        print('  cache missing e_texts_trunc — streaming court CSV to rebuild it ...')
        existing_cits = set(court_meta['e_citations'])
        e_texts_lookup = {}
        with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
            r = csv.reader(f); next(r, None)
            for row in r:
                if not row or len(row) < 2: continue
                cit = row[0]
                if cit in existing_cits:
                    e_texts_lookup[cit] = (row[1] or '')[:E_TEXT_TRUNC]
        court_meta['e_texts_trunc'] = [e_texts_lookup.get(c, '') for c in court_meta['e_citations']]
        with open(court_meta_path, 'wb') as f: pickle.dump(court_meta, f)
        print('  rebuilt e_texts_trunc and re-saved court_meta.pkl')
elif USE_COURT:
    print('Streaming court_considerations.csv (~1.99M rows) ...')
    bucket=defaultdict(list); e_cits=[]; e_jids=[]; e_texts=[]
    t0=time.time()
    E_CAP, CHAR_CAP = 30, 4000
    with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if not row or len(row) < 2: continue
            cit, text = row[0], row[1] or ''
            jid = judgment_id(cit)
            if jid is None: continue
            e_cits.append(cit); e_jids.append(jid); e_texts.append(text)
            if len(bucket[jid]) < E_CAP: bucket[jid].append(text)
    print(f'  streamed in {time.time()-t0:.1f}s  E={len(e_cits):,}  judgments={len(bucket):,}')

    judgment_ids = list(bucket.keys())
    j2idx = {j:i for i,j in enumerate(judgment_ids)}
    judgment_texts = ['\n'.join(bucket[j])[:CHAR_CAP] for j in judgment_ids]
    e_to_jidx = [j2idx[j] for j in e_jids]

    print(f'Encoding {len(judgment_texts):,} judgment-level concat texts ...')
    t0=time.time()
    j_emb = encode(judgment_texts, batch_size=32, show_progress=True)
    print(f'  judgments encoded in {time.time()-t0:.1f}s')
    j_dense = faiss.IndexFlatIP(j_emb.shape[1]); j_dense.add(j_emb); del j_emb; gc.collect()

    print(f'Encoding {len(e_texts):,} E.-level texts (~20~30 min) ...')
    t0=time.time()
    e_emb = encode(e_texts, batch_size=64, show_progress=True)
    print(f'  E-level encoded in {time.time()-t0:.1f}s')
    if len(e_emb) > 50_000:
        d = e_emb.shape[1]; nlist=4096
        quant = faiss.IndexFlatIP(d)
        e_dense = faiss.IndexIVFPQ(quant, d, nlist, 64, 8)
        e_dense.train(e_emb); e_dense.add(e_emb); e_dense.nprobe = 32
    else:
        e_dense = faiss.IndexFlatIP(e_emb.shape[1]); e_dense.add(e_emb)
    del e_emb; gc.collect()

    # truncate e_texts before pickling (1.99M × 400 chars ≈ 800 MB pickle)
    e_texts_trunc = [t[:E_TEXT_TRUNC] for t in e_texts]
    del e_texts; gc.collect()

    court_meta = {
        'judgment_ids':  judgment_ids,
        'e_citations':   e_cits,
        'e_to_jidx':     e_to_jidx,
        'e_texts_trunc': e_texts_trunc,
    }
    with open(court_meta_path, 'wb') as f: pickle.dump(court_meta, f)
    faiss.write_index(j_dense, str(court_jfaiss))
    faiss.write_index(e_dense, str(court_efaiss))
else:
    print('USE_COURT=False — skipping court indexing')
    court_meta = {'judgment_ids':[], 'e_citations':[], 'e_to_jidx':[], 'e_texts_trunc':[]}
    j_dense = None; e_dense = None

if USE_COURT:
    print(f'court ready: judgments={len(court_meta["judgment_ids"]):,}  '
          f'E-rows={len(court_meta["e_citations"]):,}  '
          f'has_text={bool(court_meta.get("e_texts_trunc"))}')


In [ ]:
# Cell 7 — EN→DE translation via NLLB (fail-soft)
TRANSLATOR = None
TRANSLATOR_TOK = None
if USE_TRANSLATE:
    NLLB_PATHS = [
        '/kaggle/input/nllb-200-distilled/nllb-200-distilled-600M',
        '/kaggle/input/nllb-200-distilled-600m',
        '/kaggle/input/nllb-200-distilled',
        'facebook/nllb-200-distilled-600M',
    ]
    for p in NLLB_PATHS:
        try:
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
            TRANSLATOR_TOK = AutoTokenizer.from_pretrained(p, src_lang='eng_Latn')
            TRANSLATOR = AutoModelForSeq2SeqLM.from_pretrained(p)
            if torch.cuda.is_available():
                TRANSLATOR = TRANSLATOR.to('cuda').half()
            print(f'NLLB loaded from {p}')
            break
        except Exception as e:
            print(f'  NLLB {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if TRANSLATOR is None:
        print('  No NLLB available — falling back to identity translation')

def translate_to_de(text):
    if TRANSLATOR is None: return text
    try:
        inputs = TRANSLATOR_TOK(text, return_tensors='pt', truncation=True, max_length=512)
        if next(TRANSLATOR.parameters()).is_cuda:
            inputs = {k:v.to('cuda') for k,v in inputs.items()}
        forced = TRANSLATOR_TOK.convert_tokens_to_ids('deu_Latn')
        with torch.no_grad():
            out = TRANSLATOR.generate(**inputs, forced_bos_token_id=forced,
                                       max_new_tokens=384, num_beams=1)
        return TRANSLATOR_TOK.batch_decode(out, skip_special_tokens=True)[0]
    except Exception:
        return text


In [ ]:
# Cell 8 — Cross-encoder reranker (BGE-reranker-v2-m3, fail-soft)
RERANKER = None
if USE_RERANKER:
    RERANK_PATHS = [
        '/kaggle/input/bge-reranker-v2-m3/bge-reranker-v2-m3',
        '/kaggle/input/bge-reranker-v2-m3',
        '/kaggle/input/baai-bge-reranker-v2-m3',
        'BAAI/bge-reranker-v2-m3',
    ]
    from sentence_transformers import CrossEncoder
    for p in RERANK_PATHS:
        try:
            RERANKER = CrossEncoder(p)
            if torch.cuda.is_available():
                RERANKER.model.to('cuda')
            print(f'Reranker loaded from {p}')
            break
        except Exception as e:
            print(f'  reranker {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if RERANKER is None:
        print('  No reranker available — fusion scores will drive ordering')

def cross_rerank(query, pairs, top_k=100):
    """pairs: [(citation, text), ...] → [(citation, score), ...] sorted desc."""
    if RERANKER is None or not pairs:
        return [(c, 0.0) for c, _ in pairs[:top_k]]
    try:
        scores = RERANKER.predict([[query, t or ''] for _, t in pairs],
                                   batch_size=32, show_progress_bar=False)
    except TypeError:
        scores = RERANKER.predict([[query, t or ''] for _, t in pairs])
    scored = list(zip([c for c,_ in pairs], (float(s) for s in scores)))
    scored.sort(key=lambda kv: -kv[1])
    return scored[:top_k]


In [ ]:
# Cell 9 — Topic classifier (priority-ordered), specific-Abs anchors, calibration

UNIVERSAL_DEFAULTS = [
    'Art. 100 Abs. 1 BGG', 'Art. 42 Abs. 1 BGG', 'Art. 42 Abs. 2 BGG',
    'Art. 95 BGG', 'Art. 106 Abs. 2 BGG',
]

# All anchors verified against laws_de.csv as exact corpus matches.
# Reasoning: parent forms expand to N children via granularity rule, but val gold
# almost always uses Abs. 1. Emitting parent → 3 children = 2 FP per anchor.
TOPIC_DEFAULTS = {
    'detention_stpo': [
        'Art. 221 Abs. 1 StPO', 'Art. 222 StPO',
        'Art. 226 Abs. 1 StPO', 'Art. 227 Abs. 1 StPO', 'Art. 228 Abs. 1 StPO',
        'Art. 229 Abs. 1 StPO', 'Art. 230 Abs. 1 StPO',
        'Art. 5 Abs. 3 BV', 'Art. 31 Abs. 1 BV', 'Art. 10 Abs. 2 BV',
        'Art. 36 Abs. 1 BV', 'Art. 5 EMRK',
        'Art. 382 Abs. 1 StPO', 'Art. 385 Abs. 1 StPO', 'Art. 390 Abs. 1 StPO',
        'Art. 393 Abs. 1 StPO', 'Art. 396 Abs. 1 StPO',
        'Art. 428 Abs. 1 StPO',
        'Art. 37 Abs. 1 StBOG', 'Art. 39 Abs. 1 StBOG',
    ],
    'family_zgb': [
        'Art. 125 Abs. 1 ZGB', 'Art. 133 Abs. 1 ZGB', 'Art. 176 Abs. 1 ZGB',
        'Art. 277 Abs. 1 ZGB', 'Art. 277 Abs. 2 ZGB',
        'Art. 285 Abs. 1 ZGB', 'Art. 286 Abs. 1 ZGB',
        'Art. 288 Abs. 1 ZGB', 'Art. 291 ZGB', 'Art. 292 ZGB',
        'Art. 308 Abs. 1 ZGB', 'Art. 311 Abs. 1 ZGB',
        'Art. 8 ZGB',
    ],
    'erbrecht_zgb': [
        'Art. 457 Abs. 1 ZGB', 'Art. 458 Abs. 1 ZGB',
        'Art. 467 ZGB', 'Art. 469 Abs. 1 ZGB', 'Art. 470 Abs. 1 ZGB',
        'Art. 471 ZGB', 'Art. 505 Abs. 1 ZGB', 'Art. 520a ZGB',
        'Art. 522 Abs. 1 ZGB',
    ],
    'mietrecht_or': [
        'Art. 257 OR', 'Art. 257d Abs. 1 OR', 'Art. 259d OR',
        'Art. 266a Abs. 1 OR', 'Art. 266g Abs. 1 OR',
        'Art. 271 Abs. 1 OR', 'Art. 271a Abs. 1 OR',
    ],
    'werkvertrag_or': [
        'Art. 363 OR', 'Art. 366 Abs. 1 OR', 'Art. 367 Abs. 1 OR',
        'Art. 368 Abs. 1 OR', 'Art. 371 Abs. 1 OR',
    ],
    'kaufvertrag_or': [
        'Art. 184 Abs. 1 OR', 'Art. 197 Abs. 1 OR', 'Art. 201 Abs. 1 OR',
        'Art. 205 Abs. 1 OR', 'Art. 208 Abs. 1 OR',
    ],
    'auftrag_or': [
        'Art. 394 Abs. 1 OR', 'Art. 398 Abs. 1 OR', 'Art. 398 Abs. 2 OR',
        'Art. 400 Abs. 1 OR', 'Art. 404 Abs. 1 OR',
    ],
    'arbeitsrecht_or': [
        'Art. 319 Abs. 1 OR', 'Art. 322 Abs. 1 OR', 'Art. 324a Abs. 1 OR',
        'Art. 335 OR', 'Art. 336 Abs. 1 OR', 'Art. 337 Abs. 1 OR',
    ],
    'strafrecht_stgb': [
        'Art. 29 Abs. 2 BV', 'Art. 32 Abs. 1 BV',
        'Art. 1 StGB', 'Art. 2 Abs. 1 StGB', 'Art. 12 Abs. 1 StGB',
        'Art. 47 Abs. 1 StGB',
    ],
    'sachenrecht_zgb': [
        'Art. 641 Abs. 1 ZGB', 'Art. 656 Abs. 1 ZGB', 'Art. 712a Abs. 1 ZGB',
        'Art. 963 Abs. 1 ZGB', 'Art. 965 Abs. 1 ZGB',
    ],
    'schuldbetreibung_schkg': [
        'Art. 17 Abs. 1 SchKG', 'Art. 82 Abs. 1 SchKG', 'Art. 83 Abs. 1 SchKG',
        'Art. 85 SchKG', 'Art. 271 Abs. 1 SchKG',
    ],
    'svg':            ['Art. 90 Abs. 1 SVG', 'Art. 91 Abs. 1 SVG', 'Art. 16 SVG'],
    'iv_ivg':         ['Art. 4 Abs. 1 IVG', 'Art. 28 Abs. 1 IVG',
                       'Art. 7 ATSG', 'Art. 8 Abs. 1 ATSG', 'Art. 16 ATSG',
                       'Art. 17 Abs. 1 IVG'],
    'dsg':            ['Art. 3 DSG', 'Art. 13 DSG', 'Art. 8 Abs. 1 DSG',
                       'Art. 13 Abs. 1 BV'],
    'urheberrecht':   ['Art. 2 URG', 'Art. 2 Abs. 1 URG', 'Art. 10 URG',
                       'Art. 10 Abs. 2 URG', 'Art. 67 Abs. 1 URG'],
    'gesellschaftsrecht': ['Art. 620 Abs. 1 OR', 'Art. 716a Abs. 1 OR',
                           'Art. 717 Abs. 1 OR', 'Art. 754 Abs. 1 OR'],
    'bger_appeal':    ['Art. 82 BGG', 'Art. 113 BGG', 'Art. 90 BGG',
                       'Art. 89 Abs. 1 BGG', 'Art. 105 Abs. 1 BGG'],
}

# Priority order: more specific topics first. The first match wins (≥1 keyword hit).
# Fixes val_005 'custody of children' getting routed to detention_stpo before family.
TOPIC_PRIORITY = [
    'family_zgb', 'erbrecht_zgb', 'mietrecht_or', 'arbeitsrecht_or',
    'werkvertrag_or', 'kaufvertrag_or', 'auftrag_or', 'gesellschaftsrecht',
    'iv_ivg', 'dsg', 'urheberrecht', 'svg',
    'schuldbetreibung_schkg', 'sachenrecht_zgb',
    'detention_stpo', 'strafrecht_stgb', 'bger_appeal',
]

TOPIC_KEYWORDS = {
    'family_zgb':     ['divorce', 'marriage', 'spouse', 'spousal', 'custody of child',
                       'custody of children', 'parental', 'co-parent', 'child support',
                       'maintenance for', 'ehegatt', 'scheidung', 'unterhalt',
                       'kinderunterhalt', 'eltern', 'familienrecht'],
    'erbrecht_zgb':   ['inheritance', 'inherit', 'estate of', 'heir', 'heirship', 'will of',
                       'testament', 'codicil', 'erbschaft', 'erbe ', 'erblasser',
                       'pflichtteil', 'erbvertrag'],
    'mietrecht_or':   ['lease', 'rental', 'tenancy', 'tenant', 'landlord', 'rent ',
                       'mietvertrag', 'mietzins', 'kündigung des miet'],
    'arbeitsrecht_or':['employment contract', 'employer', 'employee', 'wages', 'dismissal',
                       'arbeitsvertrag', 'arbeitgeber', 'arbeitnehmer', 'kündigung des arbeits',
                       'labor law', 'labour law'],
    'werkvertrag_or': ['contract for work', 'werkvertrag', 'defect of the work',
                       'baumangel', 'construction contract', 'contractor', 'unternehmer'],
    'kaufvertrag_or': ['contract of sale', 'sales contract', 'kaufvertrag', 'seller and buyer',
                       'gewährleistung'],
    'auftrag_or':     ['mandate contract', 'agency contract', 'auftrag', 'beauftragt',
                       'professional duty of care'],
    'gesellschaftsrecht': ['shareholders', 'board of directors', 'aktiengesellschaft',
                           'verwaltungsrat', 'aktionär', 'general meeting'],
    'iv_ivg':         ['invalidity', 'disability insurance', 'invalidenversicherung',
                       'iv-rente', 'erwerbsunf', 'arbeitsunf'],
    'dsg':            ['data protection', 'personal data', 'privacy', 'datenschutz',
                       'persönlichkeitsschutz'],
    'urheberrecht':   ['copyright', 'urheber', 'plagiarism', 'work of authorship'],
    'svg':            ['road traffic', 'driver', 'verkehrsregel', 'führerausweis',
                       ' svg ', 'driving licence', 'driving license', 'motor vehicle'],
    'schuldbetreibung_schkg': ['debt enforcement', 'betreibung', 'schuldbetreibung',
                               'pfändung', 'konkurs', 'rechtsöffnung', 'arrest',
                               'bankruptcy'],
    'sachenrecht_zgb':['ownership of property', 'real estate', 'eigentum', 'besitz',
                       'grundbuch', 'land register', 'grundstück'],
    'detention_stpo': ['detention', 'pre-trial detention', 'pretrial detention',
                       ' remand', 'haft', 'untersuchungshaft', 'sicherheitshaft',
                       'flight risk', 'risk of collusion', 'risk of recidivism'],
    'strafrecht_stgb':['criminal proceedings', 'criminal liability', 'theft', 'fraud',
                       'embezzlement', 'assault', 'strafgesetz', 'strafrecht',
                       'tatbestand', 'kriminell'],
    'bger_appeal':    ['appeal to the federal supreme court', 'beschwerde in zivilsachen',
                       'beschwerde in strafsachen', 'beschwerde in öffentlich',
                       'bundesgericht', 'cantonal court of last instance'],
}

TOPIC_PRIOR = {
    'family_zgb': 18, 'erbrecht_zgb': 14, 'mietrecht_or': 22,
    'arbeitsrecht_or': 22, 'werkvertrag_or': 22, 'kaufvertrag_or': 20,
    'auftrag_or': 22, 'gesellschaftsrecht': 22,
    'iv_ivg': 28, 'dsg': 18, 'urheberrecht': 18, 'svg': 22,
    'schuldbetreibung_schkg': 22, 'sachenrecht_zgb': 20,
    'detention_stpo': 38, 'strafrecht_stgb': 28, 'bger_appeal': 22,
    'default': 22,
}

def classify_topic(query):
    q = query.lower()
    for t in TOPIC_PRIORITY:
        kws = TOPIC_KEYWORDS[t]
        if any(kw in q for kw in kws):
            return t
    return 'default'

def get_defaults(topic):
    out = list(UNIVERSAL_DEFAULTS)
    if topic in TOPIC_DEFAULTS:
        out.extend(TOPIC_DEFAULTS[topic])
    return out

# Citation quality filter (carried over from phase 3 patch)
MAIN_CODES = {
    'BV', 'OR', 'ZGB', 'StGB', 'StPO', 'ZPO', 'BGG', 'SchKG', 'IVG', 'AHVG',
    'ATSG', 'KVG', 'UVG', 'AVIG', 'BVG', 'OG', 'EMRK', 'SVG', 'BetmG', 'DBG',
    'MWSTG', 'KG', 'UWG', 'URG', 'PatG', 'MSchG', 'DesG', 'AIG', 'AsylG',
    'StAhiG', 'GwG', 'FINMAG', 'BankG', 'KAG', 'VAG', 'EpG', 'TSchG', 'GSchG',
    'USG', 'NHG', 'RPG', 'WaG', 'LMG', 'HMG', 'JStG', 'JStPO', 'MStG', 'MStP',
    'ELG', 'EOG', 'FZG', 'FamZG', 'PartG', 'AHVV', 'IVV', 'KVV', 'UVV', 'BVV',
    'AVIV', 'IPRG', 'LugÜ', 'PrHG', 'KKG', 'FusG', 'DSG', 'StBOG',
}

import re as _re_sr
_SR_NUMERIC_RE = _re_sr.compile(r'\s(\d{3}(?:\.\d+)+(?:[a-z])?)\s*$')

def code_of(cit):
    parts = cit.rsplit(' ', 1)
    return parts[-1] if parts else cit

def is_sr_numeric(cit):
    return bool(_SR_NUMERIC_RE.search(cit))

def is_main_code(cit):
    return code_of(cit) in MAIN_CODES

def quality_partition(cits):
    pref, neu, dep = [], [], []
    for c in cits:
        if c.startswith('BGE ') or '_' in code_of(c):
            pref.append(c)        # BGE / BGer judgments — always prefer
        elif is_sr_numeric(c):
            dep.append(c)         # obscure SR-numbered ordinance — deprioritize
        elif is_main_code(c):
            pref.append(c)
        else:
            neu.append(c)
    return pref, neu, dep

def _elbow(sorted_scores, lo=5, hi=50):
    if len(sorted_scores) < lo + 2: return max(1, len(sorted_scores))
    arr = np.asarray(sorted_scores[:hi+1], dtype=float)
    gaps = arr[:-1] - arr[1:]
    window = gaps[lo:hi] if hi <= len(gaps) else gaps[lo:]
    if window.size == 0: return lo
    return int(np.argmax(window) + lo)

def calibrate_emit(rerank_scores, query, topic):
    sorted_scores = sorted(rerank_scores, reverse=True)
    elbow = _elbow(sorted_scores)
    prior = TOPIC_PRIOR.get(topic, TOPIC_PRIOR['default'])
    multi_issue = query.count('?') * 5
    length_bonus = (len(query.split()) // 100) * 2
    n = int(0.5 * prior + 0.4 * elbow + 0.1 * (prior + multi_issue + length_bonus))
    return max(12, min(45, n))


In [ ]:
# Cell 10 — Qwen 2.5 7B Instruct verify (fail-soft)
# Verifies the post-rerank candidate list — asks LLM which are *actually* relevant
# to the question. Output is constrained to the candidate set (we re-match the
# generated text against candidate strings; LLM can't hallucinate citations).

QWEN = None
QWEN_TOK = None
QWEN_FAILED = False

if USE_LLM:
    QWEN_PATHS = [
        '/kaggle/input/qwen25-7b-instruct/Qwen2.5-7B-Instruct',
        '/kaggle/input/qwen25-7b-instruct',
        '/kaggle/input/qwen2-5-7b-instruct',
        '/kaggle/input/qwen-2-5-7b-instruct',
        'Qwen/Qwen2.5-7B-Instruct',
    ]
    for p in QWEN_PATHS:
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            QWEN_TOK = AutoTokenizer.from_pretrained(p)
            QWEN = AutoModelForCausalLM.from_pretrained(
                p, torch_dtype=torch.float16, device_map='auto',
            )
            QWEN.eval()
            print(f'Qwen loaded from {p}')
            break
        except Exception as e:
            print(f'  qwen {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if QWEN is None:
        QWEN_FAILED = True
        print('  No Qwen available — LLM verify will be skipped at runtime')
else:
    print('USE_LLM=False — LLM verify disabled')

_NUM_LINE = re.compile(r'^\s*\(?\d+\)?[\.\)]?\s*(.+?)\s*$')

def _parse_llm_output(text, candidate_set):
    out = []
    for line in text.splitlines():
        s = line.strip()
        if not s: continue
        m = _NUM_LINE.match(s)
        if m: s = m.group(1).strip()
        # exact match first
        if s in candidate_set:
            out.append(s); continue
        # then substring match — find candidates contained in the line
        for cand in candidate_set:
            if cand in s:
                out.append(cand); break
    seen = set(); deduped = []
    for c in out:
        if c not in seen:
            seen.add(c); deduped.append(c)
    return deduped

def llm_verify(query, candidates, budget=80, max_new_tokens=900):
    """candidates: [(citation, text), ...]
    Returns: list of citations the LLM judged relevant (subset of candidate strings).
    """
    if QWEN is None or QWEN_FAILED or not candidates:
        return [c for c, _ in candidates[:budget]]
    truncated = candidates[:budget]
    cand_set = {c for c, _ in truncated}
    numbered = '\n'.join(
        f'{i+1}. {cit} :: {(text or "")[:140]}'
        for i, (cit, text) in enumerate(truncated)
    )
    prompt = (
        'You are a Swiss law expert. Given the legal question and a list of '
        'candidate citations from Swiss law / federal court, return ONLY the '
        'citation strings that are directly relevant to the question. Output '
        'one citation per line, verbatim from the list. Do NOT add anything not '
        'in the candidate list. Do NOT explain.\n\n'
        f'Question:\n{query}\n\nCandidates:\n{numbered}\n\nRelevant citations:\n'
    )
    try:
        inputs = QWEN_TOK(prompt, return_tensors='pt', truncation=True, max_length=8192)
        if next(QWEN.parameters()).is_cuda:
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        with torch.no_grad():
            out = QWEN.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, temperature=0.0,
                                 pad_token_id=QWEN_TOK.eos_token_id)
        gen = QWEN_TOK.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return _parse_llm_output(gen, cand_set)
    except Exception as e:
        print(f'  llm_verify exception: {type(e).__name__}: {str(e)[:80]}')
        return [c for c, _ in truncated]


In [ ]:
# Cell 11 — Pipeline with LLM verify integrated
RE_ART_Q  = re.compile(r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
                       r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
                       r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
                       r'\s+([A-Z][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)')
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def extract_seeds(query):
    seeds=[]
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, code = m.groups()
        parts = [f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(code)
        seeds.extend(resolve_against_corpus(' '.join(parts)))
    for m in RE_BGE_Q.finditer(query):
        vol, book, page, e = m.groups()
        base = f'BGE {vol} {book} {page}'
        if e: seeds.extend(resolve_against_corpus(f'{base} E. {e}'))
        seeds.extend(resolve_against_corpus(base))
    for m in RE_BGER_Q.finditer(query):
        case, e = m.groups()
        if e: seeds.extend(resolve_against_corpus(f'{case} E. {e}'))
        seeds.extend(resolve_against_corpus(case))
    return dedup(seeds)

_CITATION_ALIAS_RE = re.compile(
    r'\b(?:Art(?:icle)?\.?|al\.?)\s*\d+[a-z]?'
    r'(?:\s*(?:Abs\.|al\.|para\.)\s*\d+)?'
    r'(?:\s*(?:lit\.|let\.)\s*[a-z])?'
    r'\s+([A-Z][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)')

def expand_query_aliases(q):
    s = q
    for m in _CITATION_ALIAS_RE.finditer(q):
        code = m.group(1)
        if code in MULTILANG_ABBR:
            s = s + ' ' + MULTILANG_ABBR[code]
    return s

def reciprocal_rank_fusion(rankings, weights=None, k=60):
    if weights is None: weights = [1.0]*len(rankings)
    scores={}
    for rk, w in zip(rankings, weights):
        for r, cit in enumerate(rk):
            scores[cit] = scores.get(cit, 0.0) + w / (k + r + 1)
    return scores

def search_laws(query_en, query_de, top_k=200, bm25_n=500, dense_n=500):
    bm25_q = expand_query_aliases(query_de or query_en)
    bm25_scores = bm25.get_scores(tokenize(bm25_q))
    bm25_top = np.argsort(bm25_scores)[::-1][:bm25_n]
    bm25_rank = [laws_citations[i] for i in bm25_top]
    rankings=[bm25_rank]; weights=[0.30]
    q_en = encode_query(query_en)
    _, I = laws_dense.search(q_en, dense_n)
    rankings.append([laws_citations[i] for i in I[0]]); weights.append(0.60)
    if query_de and query_de != query_en:
        q_de = encode_query(query_de)
        _, I = laws_dense.search(q_de, dense_n)
        rankings.append([laws_citations[i] for i in I[0]]); weights.append(0.30)
    fused = reciprocal_rank_fusion(rankings, weights=weights)
    return sorted(fused.items(), key=lambda kv:-kv[1])[:top_k]

E_EMB_RECON = None
if USE_COURT and e_dense is not None and hasattr(e_dense, 'reconstruct_n'):
    try:
        print('Reconstructing E-level embeddings ...')
        t0=time.time()
        E_EMB_RECON = e_dense.reconstruct_n(0, e_dense.ntotal)
        print(f'  reconstructed {E_EMB_RECON.shape} in {time.time()-t0:.1f}s')
    except Exception as exc:
        print(f'  reconstruct_n failed: {exc}')

def search_court(query_en, query_de, top_k_j=60, top_k_e=200):
    if not USE_COURT or j_dense is None or e_dense is None: return []
    q_en = encode_query(query_en)
    _, I_en = j_dense.search(q_en, top_k_j)
    selected = set(int(i) for i in I_en[0] if i >= 0)
    if query_de and query_de != query_en:
        q_de = encode_query(query_de)
        _, I_de = j_dense.search(q_de, top_k_j)
        selected.update(int(i) for i in I_de[0] if i >= 0)
    e_to_jidx = court_meta['e_to_jidx']
    e_indices = [i for i, j in enumerate(e_to_jidx) if j in selected]
    if not e_indices: return []
    if E_EMB_RECON is not None:
        e_sub = E_EMB_RECON[e_indices]
    else:
        return []
    s = e_sub @ q_en[0]
    if query_de and query_de != query_en:
        s = s + e_sub @ q_de[0]
    order = np.argsort(-s)[:top_k_e]
    e_cits = court_meta['e_citations']
    return [(e_cits[e_indices[k]], float(s[k])) for k in order]

text_by_cit = dict(zip(laws_citations, laws_texts))
if USE_COURT and court_meta.get('e_texts_trunc'):
    for cit, txt in zip(court_meta['e_citations'], court_meta['e_texts_trunc']):
        if cit not in text_by_cit:
            text_by_cit[cit] = txt
print(f'text_by_cit: {len(text_by_cit):,} entries')

def predict(query, qid=None, return_debug=False):
    topic = classify_topic(query)
    seeds = extract_seeds(query)
    raw_anchors = get_defaults(topic)
    anchors = []
    for a in raw_anchors:
        resolved = resolve_against_corpus(a)
        if resolved:
            anchors.extend(resolved[:1])
    anchors = dedup(anchors)

    query_de = translate_to_de(query) if USE_TRANSLATE else query
    laws_res  = search_laws(query, query_de, top_k=200)
    court_res = search_court(query, query_de, top_k_e=200)

    merged = []
    li = ci = 0
    while li < len(laws_res) or ci < len(court_res):
        if li < len(laws_res): merged.append(laws_res[li][0]); li += 1
        if ci < len(court_res): merged.append(court_res[ci][0]); ci += 1
    cand = dedup(merged)

    if not cand:
        out = granularity_filter(dedup(seeds + anchors))[:22]
        if return_debug:
            return out, {'topic': topic, 'anchors': anchors, 'cand_pool': 0}
        return out

    pairs = [(c, text_by_cit.get(c, c)) for c in cand[:300]]
    reranked = cross_rerank(query, pairs, top_k=200)
    rerank_scores = [s for _, s in reranked]
    candidate_order = [c for c, _ in reranked]
    pref, neu, dep = quality_partition(candidate_order)

    # LLM verify on the top of pref+neu (the *plausible* part of the ranked list)
    llm_candidates = pref + neu[:max(0, 80 - len(pref))]
    llm_pairs = [(c, text_by_cit.get(c, '')) for c in llm_candidates[:80]]
    if USE_LLM and QWEN is not None:
        llm_kept = llm_verify(query, llm_pairs, budget=80)
    else:
        llm_kept = [c for c, _ in llm_pairs]

    n_emit = calibrate_emit(rerank_scores, query, topic)
    front = dedup(seeds + anchors)
    front_set = set(front)
    remaining = max(0, n_emit - len(front))

    # Fill order: LLM-kept (filtered) → leftover pref → neu → dep (SR-numeric last)
    llm_kept_set = set(llm_kept)
    leftover_pref = [c for c in pref if c not in llm_kept_set]
    pool = [c for c in (llm_kept + leftover_pref + neu + dep) if c not in front_set]
    # dedupe pool while preserving order
    seen=set(); pool_ordered=[]
    for c in pool:
        if c not in seen:
            seen.add(c); pool_ordered.append(c)
    rest = pool_ordered[:remaining]
    final = granularity_filter(dedup(front + rest))[:n_emit]

    if return_debug:
        emit_set = set(final)
        anchor_hits = [a for a in anchors if a in emit_set]
        return final, {
            'topic': topic,
            'anchors': anchors,
            'anchor_in_emit': anchor_hits,
            'anchor_miss': [a for a in anchors if a not in emit_set],
            'n_emit': n_emit,
            'cand_pool': len(cand),
            'llm_kept': len(llm_kept) if USE_LLM and QWEN is not None else None,
            'llm_input': len(llm_pairs),
        }
    return final


In [ ]:
# Cell 12 — Val Macro F1 + full diagnostic (topic, anchors, LLM keep, P/R split)
def per_query_f1(g,p):
    g,p=set(g),set(p)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp=len(g&p)
    if tp==0: return 0.0
    pr=tp/len(p); rc=tp/len(g); return 2*pr*rc/(pr+rc)
def macro_f1(gd, pd_):
    qs = set(gd)|set(pd_)
    return sum(per_query_f1(gd.get(q,[]), pd_.get(q,[])) for q in qs)/max(1,len(qs))

val = pd.read_csv(DATA/'val.csv')
gold = {r['query_id']: split_cits(r['gold_citations']) for _,r in val.iterrows()}
print(f'Predicting on val (USE_LLM={USE_LLM and QWEN is not None}) ...')
t0=time.time()
results = {}
for _, r in val.iterrows():
    pred, dbg = predict(r['query'], r['query_id'], return_debug=True)
    results[r['query_id']] = (pred, dbg)
print(f'  done in {time.time()-t0:.1f}s')

preds = {qid: p for qid, (p, _) in results.items()}
f1 = macro_f1(gold, preds)
print(f'\nVAL Macro F1 = {f1:.4f}\n')

# Per-query diagnostic
print('=== Per-query diagnostic ===')
header = f"{'qid':8s} {'topic':25s} {'F1':>6s} {'gold':>5s} {'emit':>5s} {'tp':>4s} {'anc✓/anc':>10s} {'llm':>10s}"
print(header)
for qid in sorted(gold):
    g = gold[qid]
    pred, dbg = results[qid]
    tp = len(set(g) & set(pred))
    anc = dbg.get('anchors', []); hits = dbg.get('anchor_in_emit', [])
    llm_str = f"{dbg['llm_kept']}/{dbg['llm_input']}" if dbg.get('llm_kept') is not None else 'OFF'
    print(f"{qid:8s} {dbg['topic']:25s} {per_query_f1(g, pred):6.3f} {len(g):>5d} {len(pred):>5d} {tp:>4d} {len(hits):>4d}/{len(anc):<4d}  {llm_str:>10s}")

# Anchor precision per topic
from collections import defaultdict
topic_anchor = defaultdict(lambda: {'in_gold': 0, 'total': 0})
for qid in sorted(gold):
    pred, dbg = results[qid]
    g_set = set(gold[qid])
    for a in dbg['anchors']:
        topic_anchor[dbg['topic']]['total'] += 1
        if a in g_set:
            topic_anchor[dbg['topic']]['in_gold'] += 1
print('\n=== Anchor → gold precision by topic ===')
for t, s in topic_anchor.items():
    if s['total']:
        print(f"  {t:25s}  {s['in_gold']}/{s['total']} = {s['in_gold']/s['total']:.3f}")

# Per-citation P/R: of the FPs, how many were anchors? how many came from retrieval?
print('\n=== Per-query FPs (first 5) ===')
for qid in sorted(gold):
    pred, dbg = results[qid]
    g_set = set(gold[qid]); p_set = set(pred)
    fps = [c for c in pred if c not in g_set]
    anc_set = set(dbg['anchors'])
    if fps:
        anc_fp = [c for c in fps if c in anc_set]
        ret_fp = [c for c in fps if c not in anc_set]
        print(f"  {qid}: total FP={len(fps)}  anchor-FP={len(anc_fp)}  retrieval-FP={len(ret_fp)}")
        print(f"     sample FPs: {fps[:5]}")

# Missed gold (recall side)
print('\n=== Missed gold per query (first 5 per query) ===')
for qid in sorted(gold):
    pred, dbg = results[qid]
    missed = set(gold[qid]) - set(pred)
    if missed:
        print(f"  {qid} ({dbg['topic']}) missed {len(missed)}: {list(missed)[:5]}")


In [ ]:
# Cell 12 — Generate submission.csv
test = pd.read_csv(DATA / 'test.csv')
rows=[]
print(f'Predicting on test ({len(test)} rows) ...')
t0=time.time()
for _, r in test.iterrows():
    cits = predict(r['query'], r['query_id'])
    rows.append({'query_id': r['query_id'], 'predicted_citations': ';'.join(cits)})
print(f'  done in {time.time()-t0:.1f}s')

sub = pd.DataFrame(rows)
out = WORK / 'submission.csv'
sub.to_csv(out, index=False)
print(f'\nWrote {out}  rows={len(sub)}')
print(sub.head().to_string(index=False))
counts = sub['predicted_citations'].str.count(';').add(1)
print(f'\nemit count  min={counts.min()}  mean={counts.mean():.1f}  max={counts.max()}')

# Sanity: all predictions are corpus-valid
all_preds = []
for cs in sub['predicted_citations']:
    all_preds.extend(cs.split(';'))
unique_preds = set(all_preds)
in_corpus = sum(1 for c in unique_preds if c in CORPUS)
print(f'corpus coverage: {in_corpus}/{len(unique_preds)} = {in_corpus/max(1,len(unique_preds)):.4f}')
